# Architected Material Descriptor Extraction — Exhaustive Notebook v3

**Canonical filename:** `Architected_Material_Descriptor_v3.ipynb`  
**Paired test notebook:** `Architected_Material_Descriptor_Test_v3.ipynb`

**Goal:** extract a deliberately broad structural-descriptor pool from architected materials. Feature redundancy is *allowed by design* because final descriptor selection will be performed later with an independent feature-selection workflow.

### Core concept
1. 3D CAD/STL or 2D slice stack → calibrated binary slice volume.
2. Three neighboring slices `(N, N+1, N+2)` → 8-state grayscale encoding.
3. `A = 111`, `B = 110`, `C = 011` are preserved explicitly (TSPE: Three-Slice Persistence Encoding).
4. The same stack is reconstructed as a 3D voxel representation for CT-like 3D morphometry.
5. A broad descriptor library is extracted: 2D morphology/texture, pair/triplet transitions, multi-layer persistence, topology, surface, distance/thickness, granulometry, correlation functions, lineal path, chord length, multiscale/fractal/lacunarity, spectral, directional/fabric, skeleton/network, and optional persistent homology.

### Restart-safe architecture
Every computational cell **loads its required checkpoint from disk and writes its own output**. Therefore, after a kernel restart you can resume at any completed downstream stage without relying on Python variables left in memory.

Main checkpoint folder: `WORKDIR/checkpoints/`  
Main feature folder: `WORKDIR/features/`

## Cell 01 — Bootstrap the descriptor library
This cell writes the complete helper library to disk. Run it once after receiving/copying this notebook. After that, all later cells import the saved module independently.

In [ ]:
from pathlib import Path

MAIN_NOTEBOOK_NAME = "Architected_Material_Descriptor_v3.ipynb"
TEST_NOTEBOOK_NAME = "Architected_Material_Descriptor_Test_v3.ipynb"


def _resolve_project_code_dir() -> Path:
    """Resolve the folder containing the canonical main notebook/helper modules."""
    cwd = Path.cwd().resolve()
    search_dirs = []
    for d in (cwd, cwd / "Code", cwd.parent, cwd.parent / "Code"):
        d = d.resolve()
        if d not in search_dirs and d.exists():
            search_dirs.append(d)

    for d in search_dirs:
        if (d / MAIN_NOTEBOOK_NAME).is_file():
            return d

    for d in search_dirs:
        candidates = [
            p for p in d.glob("Architected_Material_Descriptor_v3*.ipynb")
            if "test" not in p.stem.lower()
        ]
        if candidates:
            return d

    return cwd


PROJECT_CODE_DIR = _resolve_project_code_dir()
LIB_PATH = PROJECT_CODE_DIR / "descriptor_library.py"
LIB_SOURCE = 'from __future__ import annotations\nimport json, math, re, warnings\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom scipy import ndimage as ndi\nfrom scipy.stats import skew, kurtosis\nfrom skimage import measure, morphology, feature\nfrom skimage.measure import regionprops\nfrom PIL import Image\ntry:\n    import trimesh\n    _HAS_TRIMESH = True\nexcept Exception:\n    trimesh = None\n    _HAS_TRIMESH = False\n\ndef _require_trimesh(feature_name):\n    if not _HAS_TRIMESH:\n        raise RuntimeError(f"\'{feature_name}\' requires the optional \'trimesh\' package "\n                            f"(pip install trimesh). It is not installed in this environment.")\n\ntry:\n    import cadquery as _cq\n    _HAS_CADQUERY = True\nexcept Exception:\n    _cq = None\n    _HAS_CADQUERY = False\n\ndef _require_cadquery(feature_name):\n    if not _HAS_CADQUERY:\n        raise RuntimeError(f"\'{feature_name}\' requires the optional \'cadquery\' package "\n                            f"(pip install cadquery) to tessellate STEP/STP B-rep geometry into a "\n                            f"triangle mesh -- STEP files store NURBS surfaces, not triangles, so "\n                            f"trimesh cannot read them directly. It is not installed in this "\n                            f"environment. Workaround: open the file in your CAD tool and use "\n                            f"File > Export > STL, then point INPUT_PATH at that .stl instead.")\n\nEPS=1e-12\nIMAGE_EXTS={\'.png\',\'.tif\',\'.tiff\',\'.jpg\',\'.jpeg\',\'.bmp\'}\nMESH_EXTS={\'.stl\',\'.obj\',\'.ply\',\'.off\',\'.3mf\',\'.glb\',\'.gltf\'}\nSTEP_EXTS={\'.stp\',\'.step\'}\n# numpy>=2.0 renamed trapz->trapezoid and removed the old alias in later 2.x releases\n# (this broke euler_filtration_descriptors on numpy>=2.4; harmless compatibility shim).\n_trapz = getattr(np, \'trapezoid\', None) or np.trapz\nimport scipy.sparse as _sp\nfrom scipy.sparse.csgraph import dijkstra as _dijkstra\ntry:\n    import networkx as nx\n    _HAS_NETWORKX = True\nexcept Exception:\n    _HAS_NETWORKX = False\n\n# ----------------------------- IO helpers -----------------------------\ndef save_json(obj, path):\n    path=Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    with open(path,\'w\',encoding=\'utf-8\') as f: json.dump(obj,f,indent=2,ensure_ascii=False,default=_json_default)\n\ndef load_json(path):\n    with open(path,\'r\',encoding=\'utf-8\') as f: return json.load(f)\n\ndef _json_default(x):\n    if isinstance(x,(np.integer,)): return int(x)\n    if isinstance(x,(np.floating,)): return float(x)\n    if isinstance(x,np.ndarray): return x.tolist()\n    return str(x)\n\ndef save_volume(path, volume, spacing, meta=None):\n    path=Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(path, volume=volume.astype(np.uint8), spacing=np.asarray(spacing,float), meta=json.dumps(meta or {}))\n\ndef load_volume(path):\n    z=np.load(path,allow_pickle=False)\n    vol=z[\'volume\'].astype(bool); spacing=tuple(float(v) for v in z[\'spacing\'])\n    meta=json.loads(str(z[\'meta\'])) if \'meta\' in z else {}\n    return vol, spacing, meta\n\ndef natural_key(path): return [int(x) if x.isdigit() else x.lower() for x in re.split(r\'(\\d+)\',Path(path).name)]\n\ndef load_image_stack(folder, threshold=127, invert=False, spacing=(1.,1.,1.)):\n    folder=Path(folder); files=sorted([p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTS],key=natural_key)\n    if len(files)<3: raise ValueError(\'Need at least 3 slice images\')\n    arr=[]; shape=None\n    for p in files:\n        im=np.asarray(Image.open(p).convert(\'L\'))\n        if shape is None: shape=im.shape\n        elif im.shape!=shape: raise ValueError(f\'Shape mismatch: {p}\')\n        m=im>threshold\n        if invert: m=~m\n        arr.append(m)\n    return np.stack(arr).astype(bool), tuple(spacing), {\'source_type\':\'image_stack\',\'source\':str(folder),\'slice_files\':[p.name for p in files]}\n\ndef _scene_to_mesh(obj):\n    _require_trimesh(\'_scene_to_mesh\')\n    if isinstance(obj,trimesh.Scene):\n        meshes=[g for g in obj.geometry.values() if isinstance(g,trimesh.Trimesh)]\n        if not meshes: raise ValueError(\'No mesh geometry\')\n        return trimesh.util.concatenate(meshes)\n    if isinstance(obj,trimesh.Trimesh): return obj\n    raise TypeError(type(obj))\n\ndef vectorized_segments(triangles,z,eps=1e-9,vtol=1e-6):\n    """Intersect a triangle soup with the plane z=const, returning line segments.\n\n    vtol: dead-zone half-width (same length units as the vertex coordinates) that snaps any\n    vertex whose height lies within vtol of the cutting plane onto the \'<=0\' (non-crossing)\n    side before the crossing test. This is the fix for the STL \'jumping / noisy slice\'\n    artifact: an STL stores 3 independent vertex coordinates per triangle (no shared vertex\n    index), so a vertex that is geometrically shared by several triangles is written out\n    multiple times, and CAD exporters routinely leave ~1e-6..1e-9 floating-point mismatches\n    between those copies. Without vtol, a cutting plane that lands close to such a vertex can\n    see nominally-identical vertices fall on OPPOSITE sides of the raw dz<=0/dz>0 sign test,\n    which flips the parity of crossings on the affected scanline row and tears the rasterized\n    slice at a location that effectively differs at random from one slice to the next. The\n    snap only changes the *crossing classification*; the interpolated intersection point\n    itself is still computed from the true (unsnapped) coordinates, so geometric accuracy is\n    unaffected. See also repair_mesh_for_slicing(), which removes the near-duplicate vertices\n    at the source and is the first line of defense; this dead-zone is the second, independent\n    defense for cases welding cannot fully resolve (e.g. self-intersecting lattice unions).\n    """\n    if triangles.size==0: return np.zeros((0,4),float)\n    p1=triangles[:,[0,1,2],:]; p2=triangles[:,[1,2,0],:]\n    dz1=p1[:,:,2]-z; dz2=p2[:,:,2]-z\n    if vtol>0:\n        dz1=np.where(np.abs(dz1)<=vtol,-vtol,dz1)\n        dz2=np.where(np.abs(dz2)<=vtol,-vtol,dz2)\n    cross=((dz1<=0)&(dz2>0))|((dz2<=0)&(dz1>0)); valid=cross.sum(axis=1)==2\n    if not valid.any(): return np.zeros((0,4),float)\n    p1v,p2v,cv=p1[valid],p2[valid],cross[valid]\n    den=p2v[:,:,2]-p1v[:,:,2]; frac=np.zeros_like(den,float)\n    np.divide(z-p1v[:,:,2],den,out=frac,where=cv)\n    xy=p1v[:,:,:2]+frac[:,:,None]*(p2v[:,:,:2]-p1v[:,:,:2])\n    sel=xy[cv].reshape(-1,2,2); l2=((sel[:,0]-sel[:,1])**2).sum(1); sel=sel[l2>eps*eps]\n    return sel.reshape(-1,4) if sel.size else np.zeros((0,4),float)\n\ndef rasterize_segments_scanline(segments,xmin,ymin,xmax,ymax,width,height,eps=1e-12):\n    dx=(xmax-xmin)/width; dy=(ymax-ymin)/height; mask=np.zeros((height,width),bool)\n    if not len(segments): return mask,{\'odd_rows\':0,\'odd_row_frac\':0.0}\n    x1,y1,x2,y2=(segments[:,j] for j in range(4)); nh=np.abs(y2-y1)>eps; odd=0; n_active=0\n    for r in range(height):\n        y=ymax-(r+.5)*dy; a=nh&(((y1<=y)&(y<y2))|((y2<=y)&(y<y1)))\n        if not a.any(): continue\n        n_active+=1\n        xi=x1[a]+(y-y1[a])*(x2[a]-x1[a])/(y2[a]-y1[a]); xi=xi[(xi>=xmin-eps)&(xi<=xmax+eps)]; xi.sort()\n        if len(xi)%2: odd+=1; xi=xi[:-1]\n        for j in range(0,len(xi),2):\n            l,rgt=max(float(xi[j]),xmin),min(float(xi[j+1]),xmax)\n            if rgt<=l: continue\n            c0=max(0,min(width-1,int(math.ceil((l-xmin)/dx-.5)))); c1=max(0,min(width-1,int(math.floor((rgt-xmin)/dx-.5))))\n            if c1>=c0: mask[r,c0:c1+1]=True\n    return mask,{\'odd_rows\':odd,\'odd_row_frac\':(odd/n_active if n_active else 0.0)}\n\ndef _merge_vertices_safe(mesh, weld_tol):\n    """Best-effort vertex welding across trimesh API versions (digits-based rounding is the\n    most stable calling convention across trimesh releases; distance-based kwargs vary)."""\n    digits=int(max(0,min(12,round(-math.log10(max(weld_tol,1e-12))))))\n    for kwargs in ({\'digits\':digits},{}):\n        try:\n            mesh.merge_vertices(**kwargs); return True\n        except TypeError:\n            continue\n        except Exception:\n            return False\n    return False\n\ndef repair_mesh_for_slicing(mesh, voxel_size=0.2, verbose=False):\n    """Defensive mesh cleanup applied before slicing/voxelization -- the primary fix for the\n    \'jumping / noisy slice\' artifact (see vectorized_segments() docstring for the root-cause\n    explanation). Welds near-duplicate vertex copies, drops duplicate/degenerate faces, fixes\n    inconsistent face-normal winding, and attempts to close small holes so the mesh is\n    watertight going into voxelization. Returns (repaired_mesh, report_dict)."""\n    _require_trimesh(\'repair_mesh_for_slicing\')\n    report={\'watertight_before\':bool(mesh.is_watertight)}\n    weld_tol=max(voxel_size*1e-3,1e-7)\n    report[\'weld_tolerance_mm\']=weld_tol\n    report[\'vertex_weld_applied\']=_merge_vertices_safe(mesh, weld_tol)\n    for step in (\'remove_duplicate_faces\',\'remove_degenerate_faces\',\'remove_unreferenced_vertices\'):\n        try: getattr(mesh,step)()\n        except Exception: pass\n    try: trimesh.repair.fix_normals(mesh)\n    except Exception: pass\n    try: trimesh.repair.fill_holes(mesh)\n    except Exception: pass\n    report[\'watertight_after\']=bool(mesh.is_watertight)\n    report[\'n_vertices\']=int(mesh.vertices.shape[0]); report[\'n_faces\']=int(mesh.faces.shape[0])\n    if verbose:\n        print(f"[repair_mesh_for_slicing] watertight {report[\'watertight_before\']} -> {report[\'watertight_after\']}, "\n              f"V={report[\'n_vertices\']} F={report[\'n_faces\']}, weld_tol={weld_tol:.2e} mm")\n    return mesh, report\n\ndef load_step_as_mesh(step_path, voxel_size=0.2, linear_deflection=None, angular_deflection=0.3):\n    """Tessellate a STEP/STP B-rep CAD file (NURBS surfaces, not triangles) into a triangle\n    mesh via OpenCASCADE, through the optional \'cadquery\' package. STEP cannot be parsed by\n    trimesh directly -- this must run before any slicing/voxelization code. linear_deflection\n    controls tessellation fineness (mm); defaults to voxel_size/4 so the tessellation is finer\n    than the voxel grid it will be sliced into."""\n    _require_cadquery(\'load_step_as_mesh\'); _require_trimesh(\'load_step_as_mesh\')\n    if linear_deflection is None: linear_deflection=max(voxel_size/4.0,1e-4)\n    wp=_cq.importers.importStep(str(step_path))\n    solids=wp.solids().vals()\n    if not solids: solids=wp.vals()\n    if not solids: raise ValueError(f\'No solid geometry found in STEP file: {step_path}\')\n    parts=[]\n    for solid in solids:\n        verts,faces=solid.tessellate(linear_deflection,angular_deflection)\n        v=np.array([[p.x,p.y,p.z] for p in verts],float); f=np.array(faces,int)\n        parts.append(trimesh.Trimesh(vertices=v,faces=f,process=False))\n    return trimesh.util.concatenate(parts) if len(parts)>1 else parts[0]\n\ndef _legacy_scanline_voxelize(mesh, voxel_size, vtol_frac=5e-4):\n    """Plane-slicing voxelizer (the original method, now defended by the vtol dead-zone in\n    vectorized_segments). Always available given trimesh alone. Returns\n    (volume, spacing, bmin, bmax, qa_report) where qa_report lists any slices that still hit\n    an odd-parity scanline row after the fix, so residual problem slices are directly\n    identifiable."""\n    tri=np.asarray(mesh.triangles,float)\n    bmin=tri.reshape(-1,3).min(0); bmax=tri.reshape(-1,3).max(0); ext=np.maximum(bmax-bmin,voxel_size)\n    nx,ny,nz=[max(1,int(math.ceil(v/voxel_size))) for v in ext]\n    xmax,ymax,zmax=bmin+np.array([nx,ny,nz])*voxel_size\n    zc=bmin[2]+(np.arange(nz)+.5)*voxel_size\n    zmin_tri=tri[:,:,2].min(1); zmax_tri=tri[:,:,2].max(1)\n    vtol=max(voxel_size*vtol_frac,1e-7)\n    slices=[]; odd_total=0; odd_slices=[]\n    for k,z in enumerate(zc):\n        a=(zmin_tri<=z+1e-9)&(zmax_tri>z-1e-9)\n        seg=vectorized_segments(tri[a],float(z),vtol=vtol)\n        m,qa=rasterize_segments_scanline(seg,bmin[0],bmin[1],xmax,ymax,nx,ny)\n        odd_total+=qa[\'odd_rows\']\n        if qa[\'odd_rows\']>0: odd_slices.append({\'slice_index\':int(k),\'z_mm\':float(z),\'odd_rows\':int(qa[\'odd_rows\'])})\n        slices.append(m)\n    vol=np.stack(slices).astype(bool)\n    qa_report={\'odd_scanline_rows_total\':int(odd_total),\'odd_scanline_slices\':odd_slices[:50],\n               \'n_odd_slices\':len(odd_slices),\'n_slices\':int(nz),\'vtol_mm\':vtol}\n    return vol,(voxel_size,voxel_size,voxel_size),bmin,bmax,qa_report\n\ndef _trimesh_fill_voxelize(mesh, voxel_size):\n    """Flood-fill solid voxelization (trimesh.voxelized(pitch).fill()). Does not rely on\n    scanline parity at all, so it is robust to self-intersecting / non-boolean-unioned strut\n    meshes -- a common source of noisy slices in architected-lattice CAD exports that vertex\n    welding alone cannot fix. Preferred backend when available (backend=\'auto\', the default)."""\n    vg=mesh.voxelized(pitch=voxel_size).fill()\n    vol=np.asarray(vg.matrix,bool)\n    try: bmin=np.asarray(vg.bounds[0],float)\n    except Exception: bmin=np.asarray(mesh.bounds[0],float)\n    return vol,(voxel_size,voxel_size,voxel_size),bmin\n\ndef load_mesh_as_voxels(mesh_path, voxel_size=0.2, backend=\'auto\', repair=True, verbose=False):\n    """Import a mesh -- or a STEP/STP CAD file, tessellated first via load_step_as_mesh -- and\n    voxelize it for slicing-based descriptor extraction.\n\n    backend: \'auto\' (default) tries the robust flood-fill voxelizer first and sanity-checks it\n        against the mesh\'s own analytic volume, falling back to the plane-slicing scanline\n        method (now fixed, see vectorized_segments) if flood-fill is unavailable, raises, or\n        fails the sanity check. \'trimesh_fill\' or \'legacy_scanline\' force one method.\n    repair: run repair_mesh_for_slicing() (vertex welding + normal/hole repair) first. This is\n        the primary fix for the \'jumping slice\' artifact -- leave True unless the mesh is\n        already known-clean.\n    """\n    _require_trimesh(\'load_mesh_as_voxels\')\n    mesh_path=Path(mesh_path); suffix=mesh_path.suffix.lower()\n    if suffix in STEP_EXTS:\n        mesh=load_step_as_mesh(mesh_path, voxel_size=voxel_size)\n    else:\n        mesh=_scene_to_mesh(trimesh.load(mesh_path,force=\'mesh\',process=True))\n    mesh.remove_unreferenced_vertices()\n\n    repair_report={\'repaired\':False}\n    if repair:\n        mesh,repair_report=repair_mesh_for_slicing(mesh, voxel_size=voxel_size, verbose=verbose)\n        repair_report[\'repaired\']=True\n\n    qa={}; vol=None; bmin=None; bmax=None; backend_used=None\n    if backend in (\'auto\',\'trimesh_fill\'):\n        try:\n            vol,spacing,bmin=_trimesh_fill_voxelize(mesh, voxel_size)\n            tri=np.asarray(mesh.triangles,float); bmax=tri.reshape(-1,3).max(0)\n            expected_vox=(abs(float(mesh.volume))/(voxel_size**3)) if mesh.is_watertight else None\n            actual_vox=int(vol.sum())\n            sane=actual_vox>0 and (expected_vox is None or 0.2<=actual_vox/max(expected_vox,1e-9)<=5.0)\n            if not sane:\n                raise RuntimeError(f\'trimesh_fill sanity check failed: filled_voxels={actual_vox}, expected~{expected_vox}\')\n            backend_used=\'trimesh_fill\'\n            qa={\'backend_sanity_check\':\'passed\',\'expected_voxels_from_mesh_volume\':expected_vox,\'actual_filled_voxels\':actual_vox}\n        except Exception as ex:\n            if backend==\'trimesh_fill\': raise\n            qa={\'trimesh_fill_fallback_reason\':str(ex)}; vol=None\n    if vol is None:\n        vol,spacing,bmin,bmax,scan_qa=_legacy_scanline_voxelize(mesh, voxel_size)\n        backend_used=\'legacy_scanline\'; qa.update(scan_qa)\n    if bmax is None:\n        tri=np.asarray(mesh.triangles,float); bmax=tri.reshape(-1,3).max(0)\n\n    meta={\'source_type\':\'mesh\',\'source\':str(mesh_path),\'source_format\':suffix,\n          \'mesh_bounds_mm\':[np.asarray(bmin).tolist(),np.asarray(bmax).tolist()],\n          \'voxelization_backend\':backend_used,\'mesh_repair\':repair_report,\'slicing_qa\':qa,\n          \'odd_scanline_rows\':qa.get(\'odd_scanline_rows_total\',0)}\n    if verbose:\n        print(f"[load_mesh_as_voxels] backend={backend_used} shape={vol.shape} density={vol.mean():.4f}")\n        if qa.get(\'n_odd_slices\'):\n            print(f"  odd-parity rows remain on {qa[\'n_odd_slices\']}/{qa.get(\'n_slices\',\'?\')} slices after repair "\n                  f"(see meta[\'slicing_qa\'][\'odd_scanline_slices\']) -- consider backend=\'trimesh_fill\'")\n    return vol,(voxel_size,voxel_size,voxel_size),meta\n\ndef shape_based_z_interpolation(volume,spacing,factor=1):\n    if factor<=1: return volume,spacing\n    z,y,x=volume.shape; out=[]\n    for i in range(z-1):\n        a=volume[i]; b=volume[i+1]\n        da=ndi.distance_transform_edt(a)-ndi.distance_transform_edt(~a)\n        db=ndi.distance_transform_edt(b)-ndi.distance_transform_edt(~b)\n        for j in range(factor):\n            t=j/factor; out.append(((1-t)*da+t*db)>=0)\n    out.append(volume[-1]); return np.stack(out), (spacing[0]/factor,spacing[1],spacing[2])\n\ndef preprocess_volume(volume,min_component_voxels=1,fill_holes=False):\n    v=volume.astype(bool)\n    if min_component_voxels>1:\n        lab,n=ndi.label(v,structure=ndi.generate_binary_structure(3,1)); cnt=np.bincount(lab.ravel()); keep=np.where(cnt>=min_component_voxels)[0]; keep=keep[keep!=0]; v=np.isin(lab,keep)\n    if fill_holes: v=ndi.binary_fill_holes(v)\n    return v\n\n# ----------------------------- statistics -----------------------------\ndef safe_div(a,b): return float(a/b) if abs(float(b))>EPS else np.nan\n\ndef qstats(values,prefix,extra=True):\n    a=np.asarray(values,float); a=a[np.isfinite(a)]\n    keys={}\n    if a.size==0:\n        base=[\'n\',\'mean\',\'std\',\'min\',\'q01\',\'q05\',\'q10\',\'q25\',\'median\',\'q75\',\'q90\',\'q95\',\'q99\',\'max\',\'iqr\',\'range\',\'cv\',\'skew\',\'kurtosis\',\'rms\',\'mad\']\n        return {f\'{prefix}_{k}\':np.nan for k in base}\n    q=np.quantile(a,[.01,.05,.10,.25,.5,.75,.9,.95,.99])\n    mean=float(a.mean()); std=float(a.std(ddof=0)); med=float(q[4]); mad=float(np.median(np.abs(a-med)))\n    keys.update(n=int(a.size),mean=mean,std=std,min=float(a.min()),q01=float(q[0]),q05=float(q[1]),q10=float(q[2]),q25=float(q[3]),median=med,q75=float(q[5]),q90=float(q[6]),q95=float(q[7]),q99=float(q[8]),max=float(a.max()),iqr=float(q[5]-q[3]),range=float(a.max()-a.min()),cv=safe_div(std,abs(mean)),skew=float(skew(a,bias=False)) if a.size>2 else np.nan,kurtosis=float(kurtosis(a,bias=False)) if a.size>3 else np.nan,rms=float(np.sqrt(np.mean(a*a))),mad=mad)\n    return {f\'{prefix}_{k}\':v for k,v in keys.items()}\n\ndef profile_stats(values,prefix):\n    d=qstats(values,prefix); a=np.asarray(values,float); a=a[np.isfinite(a)]\n    if len(a)>=2:\n        x=np.arange(len(a)); p=np.polyfit(x,a,1); d[f\'{prefix}_slope\']=float(p[0]); d[f\'{prefix}_lag1_corr\']=float(np.corrcoef(a[:-1],a[1:])[0,1]) if np.std(a[:-1])>0 and np.std(a[1:])>0 else np.nan\n        d[f\'{prefix}_mean_abs_gradient\']=float(np.mean(np.abs(np.diff(a)))); d[f\'{prefix}_gradient_rms\']=float(np.sqrt(np.mean(np.diff(a)**2)))\n    if len(a)>=3: d[f\'{prefix}_mean_abs_second_gradient\']=float(np.mean(np.abs(np.diff(a,n=2))))\n    return d\n\ndef normalized_entropy(counts):\n    c=np.asarray(counts,float); c=c[c>0]\n    if len(c)<=1: return 0.0\n    p=c/c.sum(); return float(-(p*np.log(p)).sum()/np.log(len(c)))\n\n# ----------------------------- slice descriptors -----------------------------\ndef _largest_region(mask):\n    lab=measure.label(mask,connectivity=2); props=regionprops(lab)\n    return max(props,key=lambda r:r.area) if props else None\n\ndef slice_descriptor_table(volume,spacing):\n    dz,dy,dx=spacing; rows=[]\n    for z,m in enumerate(volume):\n        area=m.sum()*dx*dy; frac=m.mean(); per=measure.perimeter(m,neighborhood=8)*math.sqrt(dx*dy); lab=measure.label(m,connectivity=2); props=regionprops(lab)\n        eul=measure.euler_number(m,connectivity=2); lr=max(props,key=lambda r:r.area) if props else None\n        row={\'slice_index\':z,\'z_mm\':(z+.5)*dz,\'area_fraction\':frac,\'area_mm2\':area,\'perimeter_mm\':per,\'specific_perimeter_per_mm\':safe_div(per,area),\'component_count\':len(props),\'euler_number\':eul,\'hole_count_proxy\':len(props)-eul}\n        if lr:\n            cy,cx0=lr.centroid; row.update(largest_area_fraction=lr.area/m.size,largest_solidity=lr.solidity,largest_extent=lr.extent,largest_eccentricity=lr.eccentricity,largest_orientation_rad=lr.orientation,largest_equiv_diameter_mm=lr.equivalent_diameter_area*math.sqrt(dx*dy),largest_major_axis_mm=lr.axis_major_length*math.sqrt(dx*dy),largest_minor_axis_mm=lr.axis_minor_length*math.sqrt(dx*dy),centroid_x_norm=cx0/max(1,m.shape[1]-1),centroid_y_norm=cy/max(1,m.shape[0]-1),largest_circularity=safe_div(4*math.pi*lr.area,measure.perimeter(lr.image,neighborhood=8)**2))\n        else:\n            for k in [\'largest_area_fraction\',\'largest_solidity\',\'largest_extent\',\'largest_eccentricity\',\'largest_orientation_rad\',\'largest_equiv_diameter_mm\',\'largest_major_axis_mm\',\'largest_minor_axis_mm\',\'centroid_x_norm\',\'centroid_y_norm\',\'largest_circularity\']: row[k]=np.nan\n        rows.append(row)\n    df=pd.DataFrame(rows); out={}\n    for c in df.columns:\n        if c not in {\'slice_index\',\'z_mm\'}: out.update(profile_stats(df[c].values,f\'slice_{c}\'))\n    return df,out\n\ndef projection_texture_descriptors(volume):\n    out={}\n    projections={\'xy_occ\':volume.mean(0),\'xz_occ\':volume.mean(1),\'yz_occ\':volume.mean(2),\'xy_mip\':volume.max(0).astype(float),\'xz_mip\':volume.max(1).astype(float),\'yz_mip\':volume.max(2).astype(float)}\n    for name,imgf in projections.items():\n        img=np.clip(np.round(imgf*255),0,255).astype(np.uint8)\n        # quantize to 16 levels for stable GLCM\n        q=(img//16).astype(np.uint8)\n        gl=feature.graycomatrix(q,[1,2,4],[0,np.pi/4,np.pi/2,3*np.pi/4],levels=16,symmetric=True,normed=True)\n        for prop in [\'contrast\',\'dissimilarity\',\'homogeneity\',\'ASM\',\'energy\',\'correlation\']:\n            vals=feature.graycoprops(gl,prop).ravel(); out.update(qstats(vals,f\'proj_{name}_glcm_{prop}\'))\n        # image moments / Hu\n        M=measure.moments(imgf); hu=measure.moments_hu(measure.moments_normalized(measure.moments_central(imgf)))\n        for i,v in enumerate(hu): out[f\'proj_{name}_hu{i+1}\']=float(v)\n        out[f\'proj_{name}_entropy\']=float(measure.shannon_entropy(img))\n        gy,gx=np.gradient(imgf.astype(float)); gm=np.hypot(gx,gy); out.update(qstats(gm.ravel(),f\'proj_{name}_gradient\'))\n    return out\n\n# ----------------------------- pair / triplet -----------------------------\ndef pair_descriptor_table(volume,spacing):\n    dz,dy,dx=spacing; rows=[]\n    for i in range(len(volume)-1):\n        a=volume[i]; b=volume[i+1]; inter=a&b; union=a|b; ao=a&~b; bo=b&~a\n        ca=np.argwhere(a); cb=np.argwhere(b); shift=np.nan\n        if len(ca) and len(cb): shift=float(np.linalg.norm((ca.mean(0)-cb.mean(0))*np.array([dy,dx])))\n        # symmetric boundary distances\n        ba=a^ndi.binary_erosion(a); bb=b^ndi.binary_erosion(b); d_ab=d_ba=np.nan\n        if ba.any() and bb.any():\n            db=ndi.distance_transform_edt(~bb,sampling=(dy,dx)); da=ndi.distance_transform_edt(~ba,sampling=(dy,dx)); d_ab=float(db[ba].mean()); d_ba=float(da[bb].mean())\n        rows.append({\'pair_index\':i,\'lower_fraction\':a.mean(),\'upper_fraction\':b.mean(),\'intersection_fraction\':inter.mean(),\'union_fraction\':union.mean(),\'red_lower_only_fraction\':ao.mean(),\'blue_upper_only_fraction\':bo.mean(),\'jaccard\':safe_div(inter.sum(),union.sum()),\'dice\':safe_div(2*inter.sum(),a.sum()+b.sum()),\'overlap_coefficient\':safe_div(inter.sum(),min(a.sum(),b.sum())),\'containment_lower_in_upper\':safe_div(inter.sum(),a.sum()),\'containment_upper_in_lower\':safe_div(inter.sum(),b.sum()),\'symmetric_difference_fraction\':(a^b).mean(),\'signed_area_change_fraction\':b.mean()-a.mean(),\'absolute_area_change_fraction\':abs(b.mean()-a.mean()),\'centroid_shift_mm\':shift,\'boundary_mean_distance_lower_to_upper_mm\':d_ab,\'boundary_mean_distance_upper_to_lower_mm\':d_ba,\'boundary_mean_distance_symmetric_mm\':np.nanmean([d_ab,d_ba])})\n    df=pd.DataFrame(rows); out={}\n    for c in df.columns:\n        if c!=\'pair_index\': out.update(profile_stats(df[c].values,f\'pair_{c}\'))\n    return df,out\n\ndef triplet_state(a,b,c): return (a.astype(np.uint8)<<2)|(b.astype(np.uint8)<<1)|c.astype(np.uint8)\n\ndef _interface_count(label_img,p,q):\n    cnt=0\n    for ax in (0,1):\n        s1=[slice(None)]*2; s2=[slice(None)]*2; s1[ax]=slice(None,-1); s2[ax]=slice(1,None)\n        x=label_img[tuple(s1)]; y=label_img[tuple(s2)]; cnt+=np.count_nonzero(((x==p)&(y==q))|((x==q)&(y==p)))\n    return int(cnt)\n\ndef triplet_descriptor_table(volume,spacing,save_dir=None,image_stride=1):\n    dz,dy,dx=spacing; rows=[]; save_dir=Path(save_dir) if save_dir else None\n    if save_dir: save_dir.mkdir(parents=True,exist_ok=True)\n    for i in range(len(volume)-2):\n        s=triplet_state(volume[i],volume[i+1],volume[i+2]); cnt=np.bincount(s.ravel(),minlength=8); occ=cnt[1:].sum(); center=volume[i+1].sum(); abc=cnt[3]+cnt[6]+cnt[7]\n        row={\'triplet_index\':i}\n        for k in range(8): row[f\'state_{k:03b}_count\']=int(cnt[k]); row[f\'state_{k:03b}_fraction_all\']=cnt[k]/s.size\n        for k in range(1,8): row[f\'state_{k:03b}_fraction_occupied\']=safe_div(cnt[k],occ)\n        A,B,C=cnt[7],cnt[6],cnt[3]\n        row.update(A_111_fraction_all=A/s.size,B_110_fraction_all=B/s.size,C_011_fraction_all=C/s.size,ABC_fraction_all=abc/s.size,A_fraction_center=safe_div(A,center),B_fraction_center=safe_div(B,center),C_fraction_center=safe_div(C,center),two_sided_persistence_center=safe_div(A,center),adjacent_persistence_center=safe_div(A+B+C,center),growth_decay_balance=safe_div(C-B,B+C),gap_reentry_fraction_all=cnt[5]/s.size,triplet_entropy_occupied=normalized_entropy(cnt[1:]),triplet_entropy_all=normalized_entropy(cnt))\n        for state,name in [(7,\'A111\'),(6,\'B110\'),(3,\'C011\'),(5,\'gap101\')]:\n            m=s==state; lab=measure.label(m,connectivity=2); props=regionprops(lab); row[f\'{name}_component_count\']=len(props); row[f\'{name}_perimeter_px\']=measure.perimeter(m,neighborhood=8)\n            if props:\n                areas=np.array([p.area for p in props]); row[f\'{name}_largest_component_fraction\']=areas.max()/max(1,m.sum()); row[f\'{name}_component_area_cv\']=safe_div(areas.std(),areas.mean())\n            else: row[f\'{name}_largest_component_fraction\']=0.; row[f\'{name}_component_area_cv\']=np.nan\n        row[\'interface_A_B_edges\']=_interface_count(s,7,6); row[\'interface_A_C_edges\']=_interface_count(s,7,3); row[\'interface_B_C_edges\']=_interface_count(s,6,3)\n        rows.append(row)\n        if save_dir and i%image_stride==0:\n            Image.fromarray(np.round(s/7*255).astype(np.uint8)).save(save_dir/f\'triplet_{i:05d}_8state.png\')\n            abcimg=np.zeros_like(s,np.uint8); abcimg[s==7]=255; abcimg[s==6]=170; abcimg[s==3]=85; Image.fromarray(abcimg).save(save_dir/f\'triplet_{i:05d}_ABC.png\')\n    df=pd.DataFrame(rows); out={}\n    for c in df.columns:\n        if c!=\'triplet_index\': out.update(profile_stats(df[c].values,f\'triplet_{c}\'))\n    return df,out\n\ndef multilevel_persistence_descriptors(volume,max_k=7):\n    out={}; n=len(volume)\n    for k in range(2,min(max_k,n)+1):\n        vals_iou=[]; vals_min=[]; vals_center=[]\n        for i in range(n-k+1):\n            w=volume[i:i+k]; inter=np.logical_and.reduce(w); union=np.logical_or.reduce(w); occ=np.array([x.sum() for x in w]); vals_iou.append(safe_div(inter.sum(),union.sum())); vals_min.append(safe_div(inter.sum(),occ.min()))\n            vals_center.append(safe_div(inter.sum(),w[k//2].sum()))\n        out.update(profile_stats(vals_iou,f\'persist_k{k}_intersection_over_union\')); out.update(profile_stats(vals_min,f\'persist_k{k}_intersection_over_min\')); out.update(profile_stats(vals_center,f\'persist_k{k}_intersection_over_center\'))\n    return out\n\ndef lag_overlap_descriptors(volume,max_lag=12):\n    out={}; n=len(volume)\n    for lag in range(1,min(max_lag,n-1)+1):\n        jac=[]; dice=[]; sym=[]; mi=[]\n        for i in range(n-lag):\n            a=volume[i]; b=volume[i+lag]; inter=(a&b).sum(); union=(a|b).sum(); jac.append(safe_div(inter,union)); dice.append(safe_div(2*inter,a.sum()+b.sum())); sym.append((a^b).mean())\n            # binary mutual information\n            c00=np.count_nonzero(~a&~b); c01=np.count_nonzero(~a&b); c10=np.count_nonzero(a&~b); c11=np.count_nonzero(a&b); tab=np.array([[c00,c01],[c10,c11]],float); p=tab/tab.sum(); pa=p.sum(1); pb=p.sum(0); val=0\n            for x in range(2):\n                for y in range(2):\n                    if p[x,y]>0: val+=p[x,y]*math.log(p[x,y]/(pa[x]*pb[y]+EPS)+EPS)\n            mi.append(val)\n        out.update(profile_stats(jac,f\'lag{lag}_jaccard\')); out.update(profile_stats(dice,f\'lag{lag}_dice\')); out.update(profile_stats(sym,f\'lag{lag}_symdiff\')); out.update(profile_stats(mi,f\'lag{lag}_mutual_information\'))\n    # overlap-decay summaries\n    means=np.array([out.get(f\'lag{l}_jaccard_mean\',np.nan) for l in range(1,min(max_lag,n-1)+1)])\n    if np.isfinite(means).sum()>=2:\n        out[\'lag_jaccard_decay_auc\']=float(np.nansum(means)); out[\'lag_jaccard_decay_slope\']=float(np.polyfit(np.arange(1,len(means)+1)[np.isfinite(means)],means[np.isfinite(means)],1)[0])\n    return out\n\ndef triplet_transition_descriptors(volume):\n    if len(volume)<4: return {}\n    states=[triplet_state(volume[i],volume[i+1],volume[i+2]) for i in range(len(volume)-2)]; M=np.zeros((8,8),np.int64)\n    for a,b in zip(states[:-1],states[1:]):\n        idx=(a.ravel().astype(int)*8+b.ravel().astype(int)); M+=np.bincount(idx,minlength=64).reshape(8,8)\n    out={}\n    p=M/M.sum() if M.sum() else M.astype(float)\n    for i in range(8):\n        out[f\'tspe_transition_from_{i:03b}_self_probability\']=safe_div(M[i,i],M[i].sum())\n        out[f\'tspe_transition_from_{i:03b}_entropy\']=normalized_entropy(M[i])\n    out[\'tspe_transition_global_entropy\']=normalized_entropy(M.ravel()); out[\'tspe_transition_diagonal_fraction\']=safe_div(np.trace(M),M.sum()); out[\'tspe_transition_A111_persistence\']=safe_div(M[7,7],M[7].sum()); out[\'tspe_transition_B110_to_A111\']=safe_div(M[6,7],M[6].sum()); out[\'tspe_transition_A111_to_C011\']=safe_div(M[7,3],M[7].sum())\n    # full 64 transition probabilities for exhaustive feature pool\n    for i in range(8):\n        for j in range(8): out[f\'tspe_T_{i:03b}_to_{j:03b}_fraction\']=float(p[i,j]) if M.sum() else np.nan\n    return out\n\n# ----------------------------- 3D global morphology/topology -----------------------------\ndef marching_mesh(volume,spacing):\n    if not volume.any() or volume.all(): return None\n    _require_trimesh(\'marching_mesh\')\n    verts,faces,normals,vals=measure.marching_cubes(volume.astype(np.uint8),level=.5,spacing=spacing)\n    # skimage coordinates are z,y,x; convert xyz\n    verts_xyz=verts[:,[2,1,0]]; return trimesh.Trimesh(vertices=verts_xyz,faces=faces,process=False)\n\ndef morphology3d_descriptors(volume,spacing):\n    dz,dy,dx=spacing; vv=dz*dy*dx; V=volume.sum()*vv; bbox=np.array(volume.shape)*np.array(spacing); bboxV=np.prod(bbox); coords=np.argwhere(volume)\n    out={\'relative_density\':volume.mean(),\'solid_volume_mm3\':V,\'bbox_volume_mm3\':bboxV,\'bbox_fill_fraction\':safe_div(V,bboxV),\'void_fraction\':1-volume.mean(),\'bbox_x_mm\':bbox[2],\'bbox_y_mm\':bbox[1],\'bbox_z_mm\':bbox[0]}\n    if len(coords):\n        xyz=coords[:,[2,1,0]]*np.array([dx,dy,dz]); cen=xyz.mean(0); C=np.cov(xyz,rowvar=False,bias=True); eig=np.sort(np.linalg.eigvalsh(C))[::-1]; out.update(centroid_x_mm=cen[0],centroid_y_mm=cen[1],centroid_z_mm=cen[2],radius_gyration_mm=float(np.sqrt(np.trace(C))),inertia_cov_eig1=float(eig[0]),inertia_cov_eig2=float(eig[1]),inertia_cov_eig3=float(eig[2]),inertia_anisotropy_13=safe_div(eig[0],eig[2]),inertia_planarity=safe_div(eig[1]-eig[2],eig[0]),inertia_linearity=safe_div(eig[0]-eig[1],eig[0]))\n        # standardized coordinate moments\n        for ax,name in enumerate(\'xyz\'):\n            vals=xyz[:,ax]; out.update(qstats(vals,f\'coord_{name}\'))\n    try:\n        mesh=marching_mesh(volume,spacing)\n    except RuntimeError:\n        mesh=None; out[\'surface_descriptors_skipped_no_trimesh\']=1\n    if mesh is not None:\n        A=float(mesh.area); out.update(surface_area_mm2=A,specific_surface_area_per_mm=safe_div(A,V),surface_to_bbox_volume=safe_div(A,bboxV),sphericity=safe_div(math.pi**(1/3)*(6*V)**(2/3),A),compactness_36piV2_A3=safe_div(36*math.pi*V*V,A**3))\n        try:\n            hull=mesh.convex_hull; out[\'convex_hull_volume_mm3\']=float(abs(hull.volume)); out[\'convex_hull_area_mm2\']=float(hull.area); out[\'solidity_3d\']=safe_div(V,abs(hull.volume)); out[\'convexity_area_ratio\']=safe_div(hull.area,A)\n        except Exception: pass\n        # surface normal fabric and dihedral-angle statistics\n        fn=np.asarray(mesh.face_normals); fa=np.asarray(mesh.area_faces); W=fa/fa.sum(); F=np.einsum(\'i,ij,ik->jk\',W,fn,fn); ev=np.sort(np.linalg.eigvalsh(F))[::-1]\n        for i,v in enumerate(ev,1): out[f\'surface_fabric_eig{i}\']=float(v)\n        out[\'surface_fabric_fractional_anisotropy\']=float(np.sqrt(1.5*np.sum((ev-ev.mean())**2)/(np.sum(ev**2)+EPS)))\n        try:\n            ang=np.asarray(mesh.face_adjacency_angles); out.update(qstats(ang,\'surface_dihedral_angle_rad\'))\n        except Exception: pass\n    return out\n\ndef topology_descriptors(volume):\n    out={}\n    for conn,name in [(1,\'6\'),(2,\'18\'),(3,\'26\')]:\n        st=ndi.generate_binary_structure(3,conn); lab,n=ndi.label(volume,structure=st); out[f\'solid_components_conn{name}\']=int(n); vlab,vn=ndi.label(~volume,structure=st); out[f\'void_components_conn{name}\']=int(vn)\n        spans={}\n        for ax,axisname in enumerate(\'zyx\'):\n            a=np.unique(np.take(lab,0,axis=ax)); b=np.unique(np.take(lab,-1,axis=ax)); common=np.intersect1d(a[a>0],b[b>0]); spans[axisname]=len(common)\n            out[f\'solid_spanning_components_{axisname}_conn{name}\']=int(len(common)); out[f\'solid_percolates_{axisname}_conn{name}\']=float(len(common)>0)\n        # spanning material fraction\n        if n:\n            cnt=np.bincount(lab.ravel()); labels=np.arange(1,n+1); sp=set()\n            for ax in range(3):\n                a=np.unique(np.take(lab,0,axis=ax)); b=np.unique(np.take(lab,-1,axis=ax)); sp.update(np.intersect1d(a[a>0],b[b>0]).tolist())\n            out[f\'solid_spanning_voxel_fraction_conn{name}\']=safe_div(sum(cnt[list(sp)]) if sp else 0,volume.sum())\n    out[\'euler_number_conn6\']=float(measure.euler_number(volume,connectivity=1)); out[\'euler_number_conn26\']=float(measure.euler_number(volume,connectivity=3))\n    # enclosed voids: remove void connected to boundary\n    void=~volume; lab,n=ndi.label(void,structure=ndi.generate_binary_structure(3,1)); boundary=np.unique(np.concatenate([lab[0].ravel(),lab[-1].ravel(),lab[:,0].ravel(),lab[:,-1].ravel(),lab[:,:,0].ravel(),lab[:,:,-1].ravel()])); enclosed=[i for i in range(1,n+1) if i not in set(boundary.tolist())]; out[\'enclosed_void_count_conn6\']=len(enclosed)\n    if enclosed:\n        cnt=np.bincount(lab.ravel()); out.update(qstats(cnt[enclosed],\'enclosed_void_volume_vox\'))\n    # Betti proxy with cubical duality: beta0, beta2 and beta1 from Euler\n    beta0=out[\'solid_components_conn6\']; beta2=out[\'enclosed_void_count_conn6\']; chi=out[\'euler_number_conn6\']; out[\'betti0_proxy\']=beta0; out[\'betti2_proxy\']=beta2; out[\'betti1_proxy\']=float(beta0+beta2-chi)\n    return out\n\ndef euler_filtration_descriptors(volume,max_radius=6):\n    out={}; vals=[]; radii=list(range(-max_radius,max_radius+1))\n    for r in radii:\n        if r<0: m=ndi.binary_erosion(volume,iterations=-r)\n        elif r>0: m=ndi.binary_dilation(volume,iterations=r)\n        else: m=volume\n        e=float(measure.euler_number(m,connectivity=1)); vals.append(e); out[f\'euler_filtration_r{r:+d}\']=e\n    a=np.asarray(vals); out[\'euler_filtration_auc\']=float(_trapz(a,radii)); out[\'euler_filtration_range\']=float(a.max()-a.min()); out[\'euler_filtration_zero_crossings\']=int(np.count_nonzero(np.signbit(a[:-1])!=np.signbit(a[1:])))\n    return out\n\n# ----------------------------- size / thickness / granulometry -----------------------------\ndef distance_thickness_descriptors(volume,spacing):\n    sampling=spacing; ds=ndi.distance_transform_edt(volume,sampling=sampling); dv=ndi.distance_transform_edt(~volume,sampling=sampling); out={}\n    out.update(qstats(2*ds[volume],\'solid_local_thickness_proxy_mm\')); out.update(qstats(2*dv[~volume],\'void_local_diameter_proxy_mm\'))\n    try:\n        sk=morphology.skeletonize(volume); out.update(qstats(2*ds[sk],\'skeleton_sampled_solid_thickness_mm\'))\n    except Exception: pass\n    return out\n\ndef granulometry_descriptors(volume,max_radius=8):\n    out={}; base=max(1,volume.sum()); vbase=max(1,(~volume).sum())\n    for r in range(1,max_radius+1):\n        se=morphology.ball(r); op=ndi.binary_opening(volume,structure=se); cl=ndi.binary_closing(volume,structure=se); out[f\'granulo_solid_open_r{r}_retained_fraction\']=op.sum()/base; out[f\'granulo_solid_close_r{r}_volume_ratio\']=cl.sum()/base\n        vop=ndi.binary_opening(~volume,structure=se); out[f\'granulo_void_open_r{r}_retained_fraction\']=vop.sum()/vbase\n    return out\n\n# ----------------------------- chord / lineal / correlations -----------------------------\ndef _runs(line,phase=True):\n    x=(line==phase).astype(np.int16); d=np.diff(np.r_[0,x,0]); starts=np.flatnonzero(d==1); ends=np.flatnonzero(d==-1); return ends-starts\n\ndef chord_descriptors(volume,spacing):\n    out={}; names=[\'z\',\'y\',\'x\']\n    for ax,name in enumerate(names):\n        pitch=spacing[ax]; arr=np.moveaxis(volume,ax,-1).reshape(-1,volume.shape[ax])\n        for phase,pn in [(True,\'solid\'),(False,\'void\')]:\n            runs=[]\n            for line in arr: runs.extend(_runs(line,phase).tolist())\n            vals=np.asarray(runs,float)*pitch; out.update(qstats(vals,f\'chord_{pn}_{name}_mm\'))\n    return out\n\ndef lineal_path_descriptors(volume,lengths=(1,2,4,8,16,32)):\n    out={}\n    for ax,name in enumerate(\'zyx\'):\n        for phase,pn in [(True,\'solid\'),(False,\'void\')]:\n            m=volume if phase else ~volume\n            for L in lengths:\n                if L>m.shape[ax]: continue\n                # probability every voxel in a segment of length L is phase\n                c=np.ones_like(m,dtype=np.int16)\n                # sliding convolution on axis\n                sums=ndi.convolve1d(m.astype(np.int16),np.ones(L,np.int16),axis=ax,mode=\'constant\',cval=0)\n                # use centered implementation only as normalized surrogate\n                out[f\'lineal_{pn}_{name}_L{L}_survival\']=float(np.mean(sums>=L))\n    return out\n\ndef two_point_axis_descriptors(volume,max_lag=32):\n    out={}; p=volume.mean()\n    for ax,name in enumerate(\'zyx\'):\n        vals=[]\n        for lag in range(1,min(max_lag,volume.shape[ax]-1)+1):\n            s1=[slice(None)]*3; s2=[slice(None)]*3; s1[ax]=slice(None,-lag); s2[ax]=slice(lag,None); a=volume[tuple(s1)]; b=volume[tuple(s2)]; s2v=float(np.mean(a&b)); cov=s2v-p*p; norm=safe_div(cov,p*(1-p)); out[f\'two_point_{name}_lag{lag}_S2\']=s2v; out[f\'two_point_{name}_lag{lag}_normalized_cov\']=norm; vals.append(norm)\n        ar=np.asarray(vals,float); valid=np.flatnonzero(np.isfinite(ar)&(ar<=math.exp(-1)))\n        out[f\'two_point_{name}_corr_length_e1_vox\']=float(valid[0]+1) if len(valid) else np.nan\n    return out\n\ndef three_point_descriptors(volume,lags=(1,2,4,8)):\n    out={}; axes=[(0,1,\'zy\'),(0,2,\'zx\'),(1,2,\'yx\')]\n    for a1,a2,name in axes:\n        for r in lags:\n            if r>=volume.shape[a1] or r>=volume.shape[a2]: continue\n            sl0=[slice(None)]*3; sl1=[slice(None)]*3; sl2=[slice(None)]*3\n            sl0[a1]=slice(None,-r); sl0[a2]=slice(None,-r)\n            sl1[a1]=slice(r,None); sl1[a2]=slice(None,-r)\n            sl2[a1]=slice(None,-r); sl2[a2]=slice(r,None)\n            v=float(np.mean(volume[tuple(sl0)]&volume[tuple(sl1)]&volume[tuple(sl2)])); out[f\'three_point_L_{name}_lag{r}\']=v\n    return out\n\n# ----------------------------- multiscale / fractal / lacunarity -----------------------------\ndef boxcount(mask,k):\n    shape=np.array(mask.shape); trim=(shape//k)*k\n    if np.any(trim==0): return np.nan,np.nan\n    m=mask[tuple(slice(0,int(t)) for t in trim)]; new=[]\n    for n in trim: new.extend([int(n//k),k])\n    axes=list(range(0,2*mask.ndim,2)); blocks=m.reshape(new); sums=blocks.sum(axis=tuple(a+1 for a in axes)); occupied=np.count_nonzero(sums); lac=float(np.var(sums)/(np.mean(sums)**2+EPS)+1)\n    return occupied,lac\n\ndef multiscale_descriptors(volume,box_sizes=(2,3,4,6,8,12,16,24,32)):\n    out={}; xs=[]; ys=[]; boundary=volume^ndi.binary_erosion(volume)\n    for phase,name,m in [(1,\'solid\',volume),(0,\'void\',~volume),(2,\'boundary\',boundary)]:\n        X=[];Y=[]\n        for k in box_sizes:\n            occ,lac=boxcount(m,k)\n            if np.isfinite(occ): out[f\'{name}_boxcount_k{k}\']=occ; out[f\'{name}_lacunarity_k{k}\']=lac\n            if np.isfinite(occ) and occ>0: X.append(math.log(1/k)); Y.append(math.log(occ))\n        if len(X)>=2: out[f\'{name}_boxcount_fractal_dimension\']=float(np.polyfit(X,Y,1)[0])\n    return out\n\n# ----------------------------- spectral -----------------------------\ndef spectral_descriptors(volume,spacing,max_dim=128):\n    v=volume.astype(float)-volume.mean(); step=max(1,int(math.ceil(max(volume.shape)/max_dim))); vd=v[::step,::step,::step]; F=np.fft.fftn(vd); P=np.abs(F)**2; P.flat[0]=0; total=P.sum(); out={\'spectral_downsample_step\':step}\n    if total<=0: return out\n    p=P.ravel()/total; p=p[p>0]; out[\'spectral_entropy\']=float(-(p*np.log(p)).sum()/math.log(len(P.ravel())))\n    kz=np.fft.fftfreq(vd.shape[0],d=spacing[0]*step); ky=np.fft.fftfreq(vd.shape[1],d=spacing[1]*step); kx=np.fft.fftfreq(vd.shape[2],d=spacing[2]*step); KZ,KY,KX=np.meshgrid(kz,ky,kx,indexing=\'ij\'); km=np.sqrt(KX*KX+KY*KY+KZ*KZ)\n    out[\'spectral_k_mean_per_mm\']=float((P*km).sum()/total); out[\'spectral_k_rms_per_mm\']=float(np.sqrt((P*km*km).sum()/total))\n    idx=np.unravel_index(np.argmax(P),P.shape); kpeak=float(km[idx]); out[\'spectral_peak_k_per_mm\']=kpeak; out[\'spectral_peak_wavelength_mm\']=safe_div(1,kpeak)\n    # power-weighted k covariance / anisotropy\n    ks=np.stack([KX.ravel(),KY.ravel(),KZ.ravel()],1); w=P.ravel()/total; mu=(w[:,None]*ks).sum(0); C=((ks-mu)*w[:,None]).T@(ks-mu); ev=np.sort(np.linalg.eigvalsh(C))[::-1]\n    for i,e in enumerate(ev,1): out[f\'spectral_k_cov_eig{i}\']=float(e)\n    out[\'spectral_k_anisotropy_13\']=safe_div(ev[0],ev[2]); out[\'spectral_fractional_anisotropy\']=float(np.sqrt(1.5*np.sum((ev-ev.mean())**2)/(np.sum(ev**2)+EPS)))\n    # radial bins\n    kr=km.ravel(); pr=P.ravel(); edges=np.linspace(0,kr.max()+EPS,33); b=np.digitize(kr,edges)-1\n    for i in range(32):\n        mask=b==i; out[f\'spectral_radial_bin{i:02d}_fraction\']=float(pr[mask].sum()/total) if mask.any() else 0.\n    return out\n\n# ----------------------------- directional / MIL / profiles -----------------------------\ndef directional_descriptors(volume,spacing):\n    out={}; dz,dy,dx=spacing\n    # projection occupancy and gradient profile descriptors\n    for ax,name in enumerate(\'zyx\'):\n        prof=volume.mean(axis=tuple(i for i in range(3) if i!=ax)); out.update(profile_stats(prof,f\'directional_occupancy_{name}\'))\n    # mean intercept length via number of phase transitions along axis\n    for ax,name,pitch in [(0,\'z\',dz),(1,\'y\',dy),(2,\'x\',dx)]:\n        arr=np.moveaxis(volume,ax,-1).reshape(-1,volume.shape[ax]); solid_lengths=[]; void_lengths=[]; transitions=[]\n        for line in arr:\n            solid_lengths.extend((_runs(line,True)*pitch).tolist()); void_lengths.extend((_runs(line,False)*pitch).tolist()); transitions.append(np.count_nonzero(line[1:]!=line[:-1]))\n        out[f\'MIL_solid_{name}_mm\']=float(np.mean(solid_lengths)) if solid_lengths else np.nan; out[f\'MIL_void_{name}_mm\']=float(np.mean(void_lengths)) if void_lengths else np.nan; out.update(qstats(transitions,f\'line_transition_count_{name}\'))\n    vals=np.array([out.get(\'MIL_solid_x_mm\'),out.get(\'MIL_solid_y_mm\'),out.get(\'MIL_solid_z_mm\')],float); out[\'MIL_solid_max_min_ratio\']=safe_div(np.nanmax(vals),np.nanmin(vals)); out[\'MIL_solid_cv_xyz\']=safe_div(np.nanstd(vals),np.nanmean(vals))\n    return out\n\n# ----------------------------- skeleton / network -----------------------------\ndef skeleton_network_descriptors(volume,spacing,max_graph_nodes=5000):\n    out={}; sk=morphology.skeletonize(volume); n=int(sk.sum()); out[\'skeleton_voxel_count\']=n; out[\'skeleton_voxel_fraction\']=safe_div(n,volume.sum()); out[\'skeleton_length_proxy_mm\']=n*float(np.mean(spacing))\n    if not n: return out\n    kernel=np.ones((3,3,3),int); deg=ndi.convolve(sk.astype(int),kernel,mode=\'constant\')-sk.astype(int); d=deg[sk]; out.update(qstats(d,\'skeleton_voxel_degree26\')); out[\'skeleton_endpoint_fraction\']=float(np.mean(d==1)); out[\'skeleton_branchpoint_fraction\']=float(np.mean(d>=3)); out[\'skeleton_isolated_fraction\']=float(np.mean(d==0)); out[\'skeleton_degree_entropy\']=normalized_entropy(np.bincount(np.clip(d,0,26),minlength=27))\n    lab,c=ndi.label(sk,structure=ndi.generate_binary_structure(3,3)); out[\'skeleton_component_count\']=int(c)\n    # voxel-graph cycle-rank proxy from 26-neighbor adjacency\n    E=int(d.sum()/2); V=n; out[\'skeleton_graph_edge_count_proxy\']=E; out[\'skeleton_cycle_rank_proxy\']=int(E-V+c); out[\'skeleton_mean_graph_degree\']=safe_div(2*E,V)\n    # graph spectral descriptors for manageable skeletons\n    if n<=max_graph_nodes:\n        try:\n            import scipy.sparse as sp\n            from scipy.sparse.csgraph import connected_components\n            coords=np.argwhere(sk); mp={tuple(c):i for i,c in enumerate(coords)}; rr=[];cc=[]\n            neigh=[(a,b,c) for a in (-1,0,1) for b in (-1,0,1) for c in (-1,0,1) if (a,b,c)!=(0,0,0)]\n            for i,p in enumerate(coords):\n                for dv in neigh:\n                    q=tuple((p+dv).tolist()); j=mp.get(q)\n                    if j is not None and j>i: rr.extend([i,j]); cc.extend([j,i])\n            A=sp.csr_matrix((np.ones(len(rr)),(rr,cc)),shape=(n,n)); D=sp.diags(np.asarray(A.sum(1)).ravel()); L=D-A\n            if n>2:\n                from scipy.sparse.linalg import eigsh\n                k=min(8,n-1); vals=np.sort(eigsh(L,k=k,which=\'SM\',return_eigenvectors=False)); out[\'skeleton_laplacian_lambda2\']=float(vals[1]) if len(vals)>1 else np.nan; out[\'skeleton_laplacian_small_eigs_sum\']=float(vals.sum())\n                valsA=eigsh(A,k=1,which=\'LA\',return_eigenvectors=False); out[\'skeleton_adjacency_spectral_radius\']=float(valsA[0])\n        except Exception as e: out[\'skeleton_graph_spectral_skipped\']=1\n    else: out[\'skeleton_graph_spectral_skipped\']=1\n    return out\n\n# ----------------------------- optional persistent homology -----------------------------\ndef persistent_homology_descriptors(volume,max_dim_vox=64):\n    out={\'persistent_homology_available\':0}\n    try:\n        import gudhi as gd\n    except Exception:\n        return out\n    step=max(1,int(math.ceil(max(volume.shape)/max_dim_vox))); v=volume[::step,::step,::step]; f=-ndi.distance_transform_edt(v)+ndi.distance_transform_edt(~v)\n    cc=gd.CubicalComplex(top_dimensional_cells=f); pers=cc.persistence(); out[\'persistent_homology_available\']=1; out[\'persistent_homology_downsample_step\']=step\n    for dim in [0,1,2]:\n        life=[]\n        for d,(b,e) in pers:\n            if d==dim and np.isfinite(e): life.append(e-b)\n        out.update(qstats(life,f\'PH_dim{dim}_lifetime\')); out[f\'PH_dim{dim}_count\']=len(life); out[f\'PH_dim{dim}_total_persistence\']=float(np.sum(life)) if life else 0.\n    return out\n\n# ----------------------------- QA/export -----------------------------\ndef flatten_stage_summaries(stage_files):\n    row={}\n    for p in stage_files:\n        p=Path(p)\n        if not p.exists(): continue\n        if p.suffix.lower()==\'.json\':\n            d=load_json(p); row.update({k:v for k,v in d.items() if np.isscalar(v) or v is None})\n    return row\n\ndef descriptor_catalog_from_names(names):\n    rules=[\n      (\'slice_\',\'2D slice morphology\',\'Per-slice morphology and z-profile statistics\'),(\'proj_\',\'2D projection texture\',\'Orthogonal projection GLCM, Hu moments, entropy, gradients\'),(\'pair_\',\'2-layer transition\',\'Adjacent-layer overlap/change/boundary displacement\'),(\'triplet_\',\'3-layer TSPE\',\'3-layer A/B/C and 8-state statistics\'),(\'persist_\',\'multi-layer persistence\',\'Intersection persistence across k consecutive layers\'),(\'lag\',\'multi-lag interlayer\',\'Lagged overlap, mutual information and decay\'),(\'tspe_\',\'TSPE dynamics\',\'8-state transition matrix and transition entropy\'),(\'relative_density\',\'3D morphology\',\'Relative density\'),(\'solid_volume\',\'3D morphology\',\'Volume\'),(\'surface_\',\'3D surface\',\'Surface area/fabric/dihedral descriptors\'),(\'specific_surface\',\'3D surface\',\'Specific surface area\'),(\'sphericity\',\'3D shape\',\'Sphericity\'),(\'compactness\',\'3D shape\',\'Compactness\'),(\'convex_\',\'3D shape\',\'Convex-hull descriptors\'),(\'inertia_\',\'3D moments\',\'Coordinate covariance/inertia anisotropy\'),(\'coord_\',\'3D moments\',\'Coordinate distribution statistics\'),(\'solid_components\',\'3D topology\',\'Connected solid components\'),(\'void_components\',\'3D topology\',\'Connected void components\'),(\'solid_spanning\',\'3D connectivity\',\'Spanning/percolation descriptors\'),(\'euler_\',\'3D topology\',\'Euler characteristic and filtration\'),(\'betti\',\'3D topology\',\'Betti-number proxies\'),(\'enclosed_void\',\'3D topology\',\'Cavity statistics\'),(\'solid_local_thickness\',\'3D thickness\',\'Distance-transform local thickness proxy\'),(\'void_local_diameter\',\'3D pore size\',\'Distance-transform void diameter proxy\'),(\'skeleton_sampled\',\'3D thickness\',\'Thickness sampled on skeleton\'),(\'granulo_\',\'multiscale size\',\'Morphological granulometry\'),(\'chord_\',\'spatial statistics\',\'Chord-length distributions\'),(\'lineal_\',\'spatial statistics\',\'Lineal-path survival probabilities\'),(\'two_point_\',\'spatial statistics\',\'Two-point correlation\'),(\'three_point_\',\'spatial statistics\',\'Selected 3-point correlations\'),(\'solid_boxcount\',\'fractal/multiscale\',\'Solid box-counting/lacunarity\'),(\'void_boxcount\',\'fractal/multiscale\',\'Void box-counting/lacunarity\'),(\'boundary_boxcount\',\'fractal/multiscale\',\'Boundary box-counting/lacunarity\'),(\'solid_lacunarity\',\'fractal/multiscale\',\'Solid lacunarity\'),(\'void_lacunarity\',\'fractal/multiscale\',\'Void lacunarity\'),(\'spectral_\',\'frequency-domain\',\'3D FFT power spectrum, entropy, anisotropy\'),(\'directional_\',\'directionality\',\'Directional occupancy profiles\'),(\'MIL_\',\'directionality\',\'Mean intercept length\'),(\'line_transition\',\'directionality\',\'Directional phase-transition counts\'),(\'skeleton_\',\'network\',\'Skeleton morphology and graph descriptors\'),(\'PH_\',\'persistent homology\',\'Cubical persistent-homology summaries\'),\n      (\'curvature_\',\'surface curvature\',\'Discrete mean/Gaussian curvature, principal curvatures, shape index, curvedness (v4)\'),\n      (\'tortuosity_\',\'transport tortuosity\',\'Geodesic/straight-line path tortuosity per phase and axis (v4)\'),\n      (\'strut_\',\'strut/node lattice graph\',\'Junction coordination number, strut length/tortuosity/thickness distributions (v4)\'),\n      (\'soft_triplet_\',\'grayscale soft TSPE\',\'Fuzzy-logic (grayscale-preserving) A/B/C overlap, partial-volume diagnostics (v4)\'),\n      (\'ct_recon_\',\'X-ray CT reconstruction\',\'Radon-transform forward projection + filtered back-projection fidelity vs. direct stack (v4)\')]\n    rows=[]\n    for n in names:\n        fam=\'other\'; desc=\'Generated geometric descriptor\'\n        for pre,f,d in rules:\n            if n.startswith(pre): fam=f; desc=d; break\n        rows.append({\'descriptor\':n,\'family\':fam,\'description\':desc})\n    return pd.DataFrame(rows)\n\n\n# =====================================================================\n# v4 ADDITIONS -- see docstring below for the list of new descriptor families\n# =====================================================================\n"""\nDescriptor Additions v4\n========================\nNew descriptor families added on top of descriptor_library.py v3, per user request:\n\n1. curvature_descriptors           - mesh-based discrete mean/Gaussian curvature (H, K),\n                                      principal curvatures, shape index, curvedness.\n                                      (was MISSING in v3: only dihedral-angle stats existed.)\n2. tortuosity_descriptors          - geodesic/straight-line path tortuosity of solid and void\n                                      phase along z/y/x (transport-relevant; was MISSING in v3).\n3. strut_graph_descriptors         - true node/strut lattice graph: junction coordination\n                                      number Z, strut length/tortuosity/thickness-uniformity\n                                      distributions (v3 only had coarse per-voxel skeleton degree).\n4. Grayscale-preserving soft TSPE  - load_image_stack_grayscale / soft_triplet_descriptor_table:\n                                      implements the user\'s own "grayscale-based overlap" idea,\n                                      which the v3 pipeline actually short-circuited by binarizing\n                                      on load.\n5. xray_ct_projection_reconstruction_descriptors\n                                    - literal Radon-transform forward projection + filtered\n                                      back-projection (FBP) reconstruction per axial slice,\n                                      cross-validated against the direct slice-stack volume.\n                                      Implements the "use real X-ray/CT reconstruction methods"\n                                      request as an alternative/complementary extraction path.\n\nDesign constraints (kept consistent with descriptor_library.py):\n- Pure numpy/scipy/scikit-image/networkx. No new hard dependency beyond what\n  requirements.txt already lists (networkx was already required but unused by v3).\n- Every function is self-contained, side-effect-free except for optional PNG/image export,\n  and returns a flat dict of scalars so it drops into the existing restart-safe\n  checkpoint -> features/*.json -> descriptors_ALL.csv pipeline unchanged.\n- Mesh-based descriptors here use skimage.measure.marching_cubes directly (NOT trimesh),\n  so they do not require trimesh to be installed.\n"""\nimport scipy.sparse as sp\nfrom scipy.sparse.csgraph import dijkstra\n\n# =====================================================================\n# 1. CURVATURE  (mean H, Gaussian K, principal k1/k2, shape index, curvedness)\n# =====================================================================\ndef _mesh_from_volume(volume, spacing):\n    """\n    Marching-cubes mesh built directly from skimage (no trimesh dependency).\n    The volume is padded with one voxel of background on every side first: a\n    structure that touches the array boundary (a strut or lattice unit cell\n    cropped at the field of view, or a full-height rod as in the synthetic\n    test) would otherwise produce an OPEN mesh with no end cap there. Discrete\n    curvature (both the cotangent-Laplacian mean curvature and the\n    angle-defect Gaussian curvature below) assumes a closed 1-ring of\n    triangles around every vertex; at an open mesh boundary that assumption\n    is violated and produces spuriously huge curvature at the cut edge. The\n    1-voxel pad guarantees a watertight mesh so this artifact cannot occur.\n    """\n    if not volume.any() or volume.all():\n        return None, None\n    padded = np.pad(volume, 1, mode=\'constant\', constant_values=False)\n    verts, faces, normals, _ = measure.marching_cubes(padded.astype(np.uint8), level=.5, spacing=spacing)\n    verts_xyz = verts[:, [2, 1, 0]]  # skimage returns (z,y,x) -> convert to (x,y,z)\n    return verts_xyz.astype(np.float64), faces.astype(np.int64)\n\n\ndef _uniform_adjacency(n, faces):\n    edges = np.concatenate([faces[:, [0, 1]], faces[:, [1, 2]], faces[:, [2, 0]]], axis=0)\n    rows = np.concatenate([edges[:, 0], edges[:, 1]])\n    cols = np.concatenate([edges[:, 1], edges[:, 0]])\n    data = np.ones(len(rows))\n    A = sp.coo_matrix((data, (rows, cols)), shape=(n, n)).tocsr()\n    A.data[:] = 1.0  # binarize (dedupe duplicate entries from shared edges)\n    deg = np.asarray(A.sum(axis=1)).ravel()\n    return A, deg\n\n\ndef _taubin_smooth(verts, faces, iterations=12, lam=0.5, mu=-0.53):\n    """\n    Taubin (1995) lambda/mu mesh smoothing. Marching-cubes surfaces extracted\n    directly from a voxel grid carry a strong staircase artifact (locally flat\n    voxel-face terraces meeting at sharp voxel-edge ridges), which makes raw\n    per-vertex discrete curvature estimates dominated by mesh-discretization\n    noise rather than the underlying structure\'s shape. A few Taubin smoothing\n    passes remove that staircase noise while -- unlike plain Laplacian\n    smoothing -- not shrinking the surface, which is essential here since the\n    absolute curvature magnitude (not just its sign/pattern) is reported.\n    """\n    n = len(verts)\n    A, deg = _uniform_adjacency(n, faces)\n    deg_safe = np.maximum(deg, 1.0)\n    V = verts.copy()\n    for _ in range(iterations):\n        for factor in (lam, mu):\n            avg = (A @ V) / deg_safe[:, None]\n            V = V + factor * (avg - V)\n    return V\n\n\ndef _corner_geometry(verts, faces):\n    """For every (face, corner) pair, return cot(angle) at that corner and the angle itself."""\n    tri = verts[faces]  # (F,3,3)\n    corners = [(0, 1, 2), (1, 2, 0), (2, 0, 1)]\n    cots = np.zeros((len(faces), 3))\n    angles = np.zeros((len(faces), 3))\n    for k, (a, b, c) in enumerate(corners):\n        pa, pb, pc = tri[:, a], tri[:, b], tri[:, c]\n        u = pb - pa\n        v = pc - pa\n        un = np.linalg.norm(u, axis=1)\n        vn = np.linalg.norm(v, axis=1)\n        cosang = np.einsum(\'ij,ij->i\', u, v) / (un * vn + EPS)\n        cosang = np.clip(cosang, -1.0, 1.0)\n        crossn = np.linalg.norm(np.cross(u, v), axis=1)\n        sinang = crossn / (un * vn + EPS)\n        cots[:, k] = cosang / (sinang + EPS)\n        angles[:, k] = np.arccos(cosang)\n    return cots, angles, corners\n\n\ndef curvature_descriptors(volume, spacing, smooth_iterations=12):\n    """\n    Discrete differential-geometry curvature on the marching-cubes iso-surface\n    (cotangent-Laplacian mean curvature + angle-defect Gaussian curvature,\n    Meyer et al. 2003 "Discrete Differential-Geometry Operators for Triangulated\n    2-Manifolds"). Produces per-vertex H, K, principal curvatures k1/k2,\n    Koenderink shape index and curvedness, aggregated with qstats(), plus\n    area-weighted elliptic/hyperbolic/parabolic surface-fraction descriptors\n    that discriminate dome/node-like (K>0), saddle/sheet-like (K<0) and\n    cylindrical strut-like (K~0) local topology -- directly relevant to telling\n    strut-lattice vs. TPMS-sheet architected material apart.\n    """\n    out = {}\n    verts, faces = _mesh_from_volume(volume, spacing)\n    if verts is None or len(verts) < 4 or len(faces) < 4:\n        out[\'curvature_mesh_available\'] = 0\n        return out\n    out[\'curvature_mesh_available\'] = 1\n    n = len(verts)\n\n    if smooth_iterations > 0:\n        verts = _taubin_smooth(verts, faces, iterations=smooth_iterations)\n\n    # face area (also needed for mixed vertex area, barycentric 1/3-split)\n    tri = verts[faces]\n    e0 = tri[:, 1] - tri[:, 0]\n    e1 = tri[:, 2] - tri[:, 0]\n    face_area = 0.5 * np.linalg.norm(np.cross(e0, e1), axis=1)\n    area_per_vertex = np.zeros(n)\n    np.add.at(area_per_vertex, faces[:, 0], face_area / 3.0)\n    np.add.at(area_per_vertex, faces[:, 1], face_area / 3.0)\n    np.add.at(area_per_vertex, faces[:, 2], face_area / 3.0)\n    area_per_vertex = np.maximum(area_per_vertex, EPS)\n\n    cots, angles, corners = _corner_geometry(verts, faces)\n\n    # cotangent-weighted Laplacian: L(v_i) = sum_j w_ij (v_i - v_j)\n    rows = []\n    cols = []\n    vals = []\n    angle_sum = np.zeros(n)\n    for k, (a, b, c) in enumerate(corners):\n        ib = faces[:, b]\n        ic = faces[:, c]\n        w = 0.5 * cots[:, k]\n        rows.append(ib); cols.append(ic); vals.append(w)\n        rows.append(ic); cols.append(ib); vals.append(w)\n        ia = faces[:, a]\n        np.add.at(angle_sum, ia, angles[:, k])\n    rows = np.concatenate(rows); cols = np.concatenate(cols); vals = np.concatenate(vals)\n    W = sp.coo_matrix((vals, (rows, cols)), shape=(n, n)).tocsr()\n    deg = np.asarray(W.sum(axis=1)).ravel()\n    Lx = deg * verts[:, 0] - W @ verts[:, 0]\n    Ly = deg * verts[:, 1] - W @ verts[:, 1]\n    Lz = deg * verts[:, 2] - W @ verts[:, 2]\n    HN = np.stack([Lx, Ly, Lz], axis=1) / (2.0 * area_per_vertex[:, None])\n    H_mag = np.linalg.norm(HN, axis=1)\n\n    # vertex normals (area-weighted average of adjacent face normals) for sign convention\n    fn = np.cross(e0, e1)\n    fn_norm = np.linalg.norm(fn, axis=1, keepdims=True)\n    fn = fn / np.maximum(fn_norm, EPS)\n    vn = np.zeros((n, 3))\n    for k in range(3):\n        np.add.at(vn, faces[:, k], fn * face_area[:, None])\n    vn_norm = np.linalg.norm(vn, axis=1, keepdims=True)\n    vn = vn / np.maximum(vn_norm, EPS)\n\n    sign = np.sign(np.einsum(\'ij,ij->i\', HN, vn))\n    sign[sign == 0] = 1.0\n    # Sign convention fixed empirically against a synthetic solid sphere\n    # (see test_additions_v4.py section 1): H = +sign(HN . n) * |HN| gives\n    # POSITIVE mean curvature on a convex (ball) surface, matching the\n    # standard convention where a convex solid has H > 0.\n    H = sign * H_mag\n\n    K = angle_sum_defect = (2 * math.pi - angle_sum) / area_per_vertex\n\n    disc = np.maximum(H * H - K, 0.0)\n    root = np.sqrt(disc)\n    k1 = H + root\n    k2 = H - root\n    denom = (k1 - k2)\n    shape_index = np.where(np.abs(denom) > 1e-9, (2.0 / math.pi) * np.arctan2(k1 + k2, denom), 0.0)\n    curvedness = np.sqrt((k1 * k1 + k2 * k2) / 2.0)\n\n    w = area_per_vertex  # area-weighting for physically meaningful surface averages\n    out.update(qstats(H, \'curvature_mean_H_per_mm\'))\n    out.update(qstats(K, \'curvature_gaussian_K_per_mm2\'))\n    out.update(qstats(k1, \'curvature_k1_per_mm\'))\n    out.update(qstats(k2, \'curvature_k2_per_mm\'))\n    out.update(qstats(shape_index, \'curvature_shape_index\'))\n    out.update(qstats(curvedness, \'curvature_curvedness_per_mm\'))\n    out[\'curvature_area_weighted_mean_H_per_mm\'] = float(np.sum(H * w) / w.sum())\n    out[\'curvature_area_weighted_mean_K_per_mm2\'] = float(np.sum(K * w) / w.sum())\n\n    # Classification threshold: on a truly flat/cylindrical region K is theoretically\n    # exactly 0, but floating-point round-off leaves it at ~1e-9..1e-12, so a threshold\n    # relative to median(|K|) (which itself can be ~0 when most of the surface is flat)\n    # mis-splits that noise ~50/50 into spurious "elliptic"/"hyperbolic" labels. A fixed\n    # small absolute threshold in [1/spacing-unit]^2 comfortably separates float noise\n    # (~1e-9) from any physically meaningful curvature at typical voxel/mm scales\n    # (K >~ 1e-3 for sub-metre radii of curvature). If your spacing units are far from\n    # ~1 (e.g. sub-micron voxels in metres), rescale CURVATURE_FLAT_EPS accordingly.\n    k_eps = 1e-6\n    ell = w[K > k_eps].sum(); hyp = w[K < -k_eps].sum(); par = w[np.abs(K) <= k_eps].sum()\n    tot = w.sum()\n    out[\'curvature_area_fraction_elliptic_dome_or_node\'] = safe_div(ell, tot)\n    out[\'curvature_area_fraction_hyperbolic_saddle_or_sheet\'] = safe_div(hyp, tot)\n    out[\'curvature_area_fraction_parabolic_cylindrical_or_strut\'] = safe_div(par, tot)\n    return out\n\n\n# =====================================================================\n# 2. TORTUOSITY  (geodesic / straight-line path length ratio, per phase & axis)\n# =====================================================================\ndef _largest_component(mask, connectivity=1):\n    st = ndi.generate_binary_structure(3, connectivity)\n    lab, n = ndi.label(mask, structure=st)\n    if n == 0:\n        return None\n    sizes = ndi.sum(mask, lab, index=np.arange(1, n + 1))\n    biggest = 1 + int(np.argmax(sizes))\n    return lab == biggest\n\n\ndef _build_voxel_graph(mask, spacing):\n    """6-connected weighted sparse adjacency graph over voxels of `mask`."""\n    Z, Y, X = mask.shape\n    idx = -np.ones((Z, Y, X), dtype=np.int64)\n    coords = np.argwhere(mask)\n    idx[tuple(coords.T)] = np.arange(len(coords))\n    rows, cols, data = [], [], []\n    for (dz, dy, dx) in [(1, 0, 0), (0, 1, 0), (0, 0, 1)]:\n        s1 = (slice(0, Z - dz or None), slice(0, Y - dy or None), slice(0, X - dx or None))\n        s2 = (slice(dz, Z), slice(dy, Y), slice(dx, X))\n        m = mask[s1] & mask[s2]\n        i1 = idx[s1][m]\n        i2 = idx[s2][m]\n        w = math.sqrt((dz * spacing[0]) ** 2 + (dy * spacing[1]) ** 2 + (dx * spacing[2]) ** 2)\n        rows.append(i1); cols.append(i2); data.append(np.full(i1.shape, w))\n    if rows:\n        rows = np.concatenate(rows); cols = np.concatenate(cols); data = np.concatenate(data)\n    else:\n        rows = np.array([], dtype=np.int64); cols = np.array([], dtype=np.int64); data = np.array([])\n    n = len(coords)\n    G = sp.coo_matrix((data, (rows, cols)), shape=(n, n)).tocsr()\n    G = G.maximum(G.T)\n    return G, coords, idx\n\n\ndef _axis_tortuosity(mask, spacing, axis, max_side=48):\n    factor = max(1, int(math.ceil(max(mask.shape) / max_side)))\n    if factor > 1:\n        mask = mask[::factor, ::factor, ::factor]\n        spacing = tuple(s * factor for s in spacing)\n    comp = _largest_component(mask)\n    if comp is None or comp.sum() < 4:\n        return np.nan\n    lo = 0\n    hi = comp.shape[axis] - 1\n    sl_lo = [slice(None)] * 3; sl_lo[axis] = 0\n    sl_hi = [slice(None)] * 3; sl_hi[axis] = hi\n    src_mask = np.zeros_like(comp); src_mask[tuple(sl_lo)] = comp[tuple(sl_lo)]\n    dst_mask = np.zeros_like(comp); dst_mask[tuple(sl_hi)] = comp[tuple(sl_hi)]\n    if not src_mask.any() or not dst_mask.any():\n        return np.nan  # phase does not span/percolate along this axis\n    G, coords, idx = _build_voxel_graph(comp, spacing)\n    src_idx = idx[src_mask]\n    src_idx = src_idx[src_idx >= 0]\n    dst_idx = idx[dst_mask]\n    dst_idx = dst_idx[dst_idx >= 0]\n    if len(src_idx) == 0 or len(dst_idx) == 0:\n        return np.nan\n    dist = dijkstra(G, directed=False, indices=src_idx, min_only=True)\n    d_at_dst = dist[dst_idx]\n    d_at_dst = d_at_dst[np.isfinite(d_at_dst)]\n    if len(d_at_dst) == 0:\n        return np.nan\n    geodesic_mean = float(np.mean(d_at_dst))\n    straight = hi * spacing[axis]\n    return safe_div(geodesic_mean, straight)\n\n\ndef tortuosity_descriptors(volume, spacing, max_side=48):\n    """\n    Geodesic tortuosity tau = <L_geodesic> / L_straight of the LARGEST connected\n    component of each phase, from the low-coordinate face to the high-coordinate\n    face along each axis (multi-source Dijkstra on the 6-connected voxel graph,\n    downsampled to <= max_side per dimension for tractability). NaN when the\n    phase does not span/percolate that axis (mirrors the existing\n    `solid_percolates_*` flags in topology_descriptors -- tortuosity is only\n    defined for a spanning pathway). Void tortuosity is the standard transport\n    descriptor for permeability/diffusivity (Kozeny-Carman-type estimates);\n    solid tortuosity is the analogous descriptor for conduction path length\n    (thermal/electrical) through the strut network.\n    """\n    out = {}\n    for phase_mask, pname in [(volume, \'solid\'), (~volume, \'void\')]:\n        for ax, axname in enumerate(\'zyx\'):\n            out[f\'tortuosity_{pname}_{axname}\'] = _axis_tortuosity(phase_mask, spacing, ax, max_side)\n        vals = np.array([out[f\'tortuosity_{pname}_{a}\'] for a in \'zyx\'], float)\n        out[f\'tortuosity_{pname}_mean_xyz\'] = float(np.nanmean(vals)) if np.isfinite(vals).any() else np.nan\n    return out\n\n\n# =====================================================================\n# 3. STRUT / NODE GRAPH DESCRIPTORS (true lattice topology)\n# =====================================================================\ndef strut_graph_descriptors(volume, spacing, max_skeleton_voxels=250000):\n    """\n    Reduces the 26-connected skeleton voxel graph to the actual node/strut\n    lattice graph used in architected-material literature: junction nodes\n    (skeleton voxels of degree != 2) connected by struts (chains of degree-2\n    voxels). Reports nodal coordination number Z (Maxwell isostaticity\n    reference: Z=6 for a 3D central-force frame, Z=4 in 2D), strut\n    geodesic/straight length, per-strut tortuosity, and per-strut thickness\n    (sampled from the solid distance-transform along the strut path) including\n    a thickness-uniformity CV -- descriptors the v3 skeleton family did not\n    have (it only reported local, per-voxel degree statistics).\n    """\n    out = {}\n    if not _HAS_NETWORKX:\n        out[\'strut_graph_skipped_no_networkx\'] = 1\n        return out\n    sk = morphology.skeletonize(volume)\n    n_sk = int(sk.sum())\n    if n_sk == 0:\n        out[\'strut_graph_skipped_empty\'] = 1\n        return out\n    if n_sk > max_skeleton_voxels:\n        out[\'strut_graph_skipped_too_large\'] = 1\n        return out\n\n    coords = np.argwhere(sk)\n    coord_set = set(map(tuple, coords.tolist()))\n    G = nx.Graph()\n    neighbors26 = [(a, b, c) for a in (-1, 0, 1) for b in (-1, 0, 1) for c in (-1, 0, 1) if (a, b, c) != (0, 0, 0)]\n    for p in coords:\n        pt = tuple(int(v) for v in p)\n        G.add_node(pt)\n    for p in coords:\n        pt = tuple(int(v) for v in p)\n        for d in neighbors26:\n            q = (pt[0] + d[0], pt[1] + d[1], pt[2] + d[2])\n            if q in coord_set and q > pt:\n                w = math.sqrt(sum((di * si) ** 2 for di, si in zip(d, spacing)))\n                G.add_edge(pt, q, weight=w)\n\n    deg = dict(G.degree())\n    junction_nodes = [p for p, d in deg.items() if d != 2]\n\n    def phys(p):\n        return np.array([p[i] * spacing[i] for i in range(3)])\n\n    visited_edges = set()\n    struts = []\n    for jn in junction_nodes:\n        for nb in list(G.neighbors(jn)):\n            e = frozenset((jn, nb))\n            if e in visited_edges:\n                continue\n            visited_edges.add(e)\n            path = [jn, nb]\n            length = G[jn][nb][\'weight\']\n            prev, cur = jn, nb\n            steps = 0\n            while deg.get(cur, 0) == 2 and steps < n_sk + 5:\n                nxts = [x for x in G.neighbors(cur) if x != prev]\n                if not nxts:\n                    break\n                nxt = nxts[0]\n                e2 = frozenset((cur, nxt))\n                if e2 in visited_edges:\n                    break\n                visited_edges.add(e2)\n                length += G[cur][nxt][\'weight\']\n                path.append(nxt)\n                prev, cur = cur, nxt\n                steps += 1\n            end = cur\n            straight = float(np.linalg.norm(phys(jn) - phys(end)))\n            struts.append({\n                \'start\': jn, \'end\': end, \'geodesic_length\': float(length),\n                \'straight_length\': straight,\n                \'tortuosity\': safe_div(length, straight) if straight > EPS else np.nan,\n                \'path\': path,\n            })\n\n    dedup = {}\n    for s in struts:\n        key = (frozenset((s[\'start\'], s[\'end\'])), round(s[\'geodesic_length\'], 6), len(s[\'path\']))\n        dedup[key] = s\n    struts = list(dedup.values())\n\n    node_degrees = np.array([deg[jn] for jn in junction_nodes], float)\n    out[\'strut_node_count\'] = int(len(junction_nodes))\n    out[\'strut_endpoint_node_count\'] = int(np.sum(node_degrees == 1))\n    out[\'strut_junction_node_count\'] = int(np.sum(node_degrees >= 3))\n    junction_only = node_degrees[node_degrees >= 3]\n    out.update(qstats(junction_only, \'strut_node_coordination_number_Z\'))\n    out[\'strut_count\'] = int(len(struts))\n    out[\'strut_network_mean_coordination_number_Z\'] = safe_div(2 * len(struts), max(1, len(junction_nodes)))\n\n    if struts:\n        lengths = np.array([s[\'geodesic_length\'] for s in struts])\n        straight = np.array([s[\'straight_length\'] for s in struts])\n        tort = np.array([s[\'tortuosity\'] for s in struts])\n        out.update(qstats(lengths, \'strut_geodesic_length_mm\'))\n        out.update(qstats(straight, \'strut_straight_length_mm\'))\n        out.update(qstats(tort, \'strut_tortuosity\'))\n\n        ds = ndi.distance_transform_edt(volume, sampling=spacing)\n        means, cvs, minmax = [], [], []\n        for s in struts:\n            pth = np.array(s[\'path\'])\n            vals = 2.0 * ds[tuple(pth.T)]\n            if len(vals) == 0:\n                continue\n            m = float(vals.mean())\n            means.append(m)\n            cvs.append(safe_div(float(vals.std()), m))\n            mx = float(vals.max())\n            minmax.append(safe_div(float(vals.min()), mx) if mx > EPS else np.nan)\n        out.update(qstats(means, \'strut_thickness_mean_mm\'))\n        out.update(qstats(cvs, \'strut_thickness_uniformity_cv\'))\n        out.update(qstats(minmax, \'strut_thickness_min_over_max\'))\n        out[\'strut_aspect_ratio_median\'] = safe_div(float(np.median(lengths)), float(np.median(means))) if means else np.nan\n    return out\n\n\n# =====================================================================\n# 4. GRAYSCALE-PRESERVING TSPE (soft/fuzzy A/B/C overlap)\n# =====================================================================\ndef load_image_stack_grayscale(folder, spacing=(1., 1., 1.), invert=False):\n    """\n    Loads the same slice series as load_image_stack but WITHOUT binarizing:\n    keeps normalized [0,1] grayscale intensity per voxel. This is what actually\n    implements the user\'s stated concept ("grayscale 기반으로 겹치는 부분 결정")\n    -- the v3 pipeline binarized on load (load_image_stack: `im>threshold`)\n    before any A/B/C overlap logic ever saw the pixels, so the "grayscale" idea\n    never reached the descriptor stage in the original code.\n    """\n    folder = Path(folder)\n    files = sorted([p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTS], key=natural_key)\n    if len(files) < 3:\n        raise ValueError(\'Need at least 3 slice images\')\n    arr = []\n    shape = None\n    for p in files:\n        im = np.asarray(Image.open(p).convert(\'L\')).astype(np.float32)\n        if shape is None:\n            shape = im.shape\n        elif im.shape != shape:\n            raise ValueError(f\'Shape mismatch: {p}\')\n        if invert:\n            im = 255.0 - im\n        arr.append(im / 255.0)\n    vol = np.stack(arr).astype(np.float32)\n    return vol, tuple(spacing), {\'source_type\': \'image_stack_grayscale\', \'source\': str(folder),\n                                  \'slice_files\': [p.name for p in files]}\n\n\ndef grayscale_stack_from_binary(volume, blur_sigma=0.0):\n    """\n    Fallback path when only a binary volume checkpoint exists (e.g. mesh input,\n    or a v3 run already binarized on load): synthesizes a soft [0,1] field by\n    optionally blurring the binary volume, so the soft-TSPE math below stays\n    exercisable even without re-reading the original grayscale images. When\n    blur_sigma=0 this is numerically identical to the hard TSPE (A/B/C soft ==\n    A/B/C hard), which is used as the correctness check in the test script.\n    """\n    v = volume.astype(np.float32)\n    if blur_sigma > 0:\n        v = ndi.gaussian_filter(v, sigma=blur_sigma)\n    return v\n\n\ndef save_grayscale_stack(path, gvol, spacing, meta=None):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(path, volume=gvol.astype(np.float32), spacing=np.asarray(spacing, float),\n                         meta=json.dumps(meta or {}))\n\n\ndef load_grayscale_stack(path):\n    z = np.load(path, allow_pickle=False)\n    vol = z[\'volume\'].astype(np.float32)\n    spacing = tuple(float(v) for v in z[\'spacing\'])\n    meta = json.loads(str(z[\'meta\'])) if \'meta\' in z else {}\n    return vol, spacing, meta\n\n\ndef soft_triplet_descriptor_table(gvol, spacing, save_dir=None, image_stride=1, cut=0.5):\n    """\n    Continuous-valued counterpart of triplet_descriptor_table(): for\n    consecutive grayscale slices I_N, I_{N+1}, I_{N+2} in [0,1], computes the\n    fuzzy-logic (Zadeh min t-norm) equivalents of A/B/C:\n        A_soft = min(I_N, I_{N+1}, I_{N+2})\n        B_soft = min(I_N, I_{N+1}, 1 - I_{N+2})\n        C_soft = min(I_{N+1}, I_{N+2}, 1 - I_N)\n    min() is used (rather than the product t-norm) because it is the standard\n    fuzzy-AND and keeps A_soft/B_soft/C_soft bounded by each slice\'s own\n    intensity -- i.e. an overlap region can be no "stronger" than its weakest\n    constituent slice, which is the same physical idea as partial-volume CT\n    voxels: a voxel that is only 40% solid in one of the three layers cannot\n    contribute more than 0.4 to a persistence measure spanning that layer.\n    Also reports the information lost by hard-thresholding\n    (soft_A_hard_agreement_dice) and the fraction of genuinely ambiguous\n    partial-volume pixels (soft_A_partial_volume_fraction).\n    """\n    dz, dy, dx = spacing\n    rows = []\n    save_dir = Path(save_dir) if save_dir else None\n    if save_dir:\n        save_dir.mkdir(parents=True, exist_ok=True)\n    for i in range(len(gvol) - 2):\n        I0, I1, I2 = gvol[i], gvol[i + 1], gvol[i + 2]\n        A_soft = np.minimum(np.minimum(I0, I1), I2)\n        B_soft = np.minimum(np.minimum(I0, I1), 1 - I2)\n        C_soft = np.minimum(np.minimum(I1, I2), 1 - I0)\n        row = {\'triplet_index\': i}\n        for name, img in [(\'A\', A_soft), (\'B\', B_soft), (\'C\', C_soft)]:\n            row.update(qstats(img.ravel(), f\'soft_{name}\'))\n        hardA = (I0 > cut) & (I1 > cut) & (I2 > cut)\n        pred_softA = A_soft > cut\n        inter = np.count_nonzero(pred_softA & hardA)\n        row[\'soft_A_hard_agreement_dice\'] = safe_div(2 * inter, pred_softA.sum() + hardA.sum())\n        row[\'soft_A_partial_volume_fraction\'] = float(np.mean((A_soft > 0.02) & (A_soft < 0.98)))\n        row[\'soft_B_partial_volume_fraction\'] = float(np.mean((B_soft > 0.02) & (B_soft < 0.98)))\n        row[\'soft_C_partial_volume_fraction\'] = float(np.mean((C_soft > 0.02) & (C_soft < 0.98)))\n        rows.append(row)\n        if save_dir and i % image_stride == 0:\n            comp = np.stack([np.clip(A_soft * 255, 0, 255), np.clip(B_soft * 255, 0, 255),\n                              np.clip(C_soft * 255, 0, 255)], axis=-1).astype(np.uint8)\n            Image.fromarray(comp).save(save_dir / f\'soft_triplet_{i:05d}_ABC_rgb.png\')\n    df = pd.DataFrame(rows)\n    out = {}\n    for c in df.columns:\n        if c != \'triplet_index\':\n            out.update(profile_stats(df[c].values, f\'soft_triplet_{c}\'))\n    return df, out\n\n\n# =====================================================================\n# 5. X-RAY CT PROJECTION SIMULATION + FILTERED BACK-PROJECTION RECONSTRUCTION\n# =====================================================================\ndef xray_ct_projection_reconstruction(volume, spacing, n_angles=180, sparse_n_angles=60,\n                                       add_poisson_noise=False, photon_count=5000.0, mu=0.15,\n                                       seed=0):\n    """\n    Simulates an actual X-ray CT acquisition + reconstruction of this structure,\n    independent of the direct slice-stack pipeline: for every axial (z) slice,\n    forward-projects with the Radon transform (skimage.transform.radon,\n    parallel-beam) at `n_angles` evenly spaced angles in [0,180), optionally\n    injects Poisson photon-counting noise (Beer-Lambert attenuation model\n    I = I0*exp(-mu*x)), then reconstructs with filtered back-projection\n    (skimage.transform.iradon, ramp filter -- the same FBP algorithm described\n    in the accompanying discussion). A second `sparse_n_angles` run demonstrates\n    the effect of a lower-dose / fewer-projection acquisition.\n\n    This is the concrete implementation of "use real X-ray/CT reconstruction\n    techniques as an alternative 3D-structure extraction pathway": a user who\n    only has raw radiographic projections (rather than already-registered\n    slice images) can run their projections through the same\n    `xray_ct_projection_reconstruction` call, feed the resulting reconstructed\n    volume back into every descriptor function in this library, and skip the\n    slice-stack step entirely.\n\n    Returns (descriptors, recon_full_volume, recon_sparse_volume). Descriptors\n    include reconstruction fidelity vs. the direct-stack ground truth (Dice,\n    IoU, relative-density error) at both angle counts.\n    """\n    rng = np.random.default_rng(seed)\n    from skimage.transform import radon, iradon\n    Z, Y, X = volume.shape\n    theta_full = np.linspace(0., 180., n_angles, endpoint=False)\n    theta_sparse = np.linspace(0., 180., sparse_n_angles, endpoint=False)\n    recon_full = np.zeros_like(volume, dtype=bool)\n    recon_sparse = np.zeros_like(volume, dtype=bool)\n\n    def _project_and_reconstruct(sl, theta):\n        sino = radon(sl.astype(float), theta=theta, circle=False)\n        if add_poisson_noise:\n            atten = np.clip(sino, 0, None)\n            transmitted = photon_count * np.exp(-atten * mu)\n            noisy = rng.poisson(np.clip(transmitted, 1, None)).astype(float)\n            sino = -np.log(np.clip(noisy, 1, None) / photon_count) / mu\n        rec = iradon(sino, theta=theta, filter_name=\'ramp\', circle=False, output_size=sl.shape[0])\n        thr = 0.5 * rec.max() if rec.max() > 0 else 0.5\n        return rec > thr\n\n    for z in range(Z):\n        sl = volume[z]\n        if not sl.any():\n            continue\n        recon_full[z] = _project_and_reconstruct(sl, theta_full)\n        recon_sparse[z] = _project_and_reconstruct(sl, theta_sparse)\n\n    out = {\'ct_recon_n_angles_full\': n_angles, \'ct_recon_n_angles_sparse\': sparse_n_angles,\n           \'ct_recon_poisson_noise_applied\': int(add_poisson_noise)}\n    for tag, rec in [(\'full_angle\', recon_full), (\'sparse_angle\', recon_sparse)]:\n        inter = int(np.count_nonzero(rec & volume))\n        union = int(np.count_nonzero(rec | volume))\n        out[f\'ct_recon_{tag}_dice_vs_direct_stack\'] = safe_div(2 * inter, int(rec.sum()) + int(volume.sum()))\n        out[f\'ct_recon_{tag}_iou_vs_direct_stack\'] = safe_div(inter, union)\n        out[f\'ct_recon_{tag}_relative_density\'] = float(rec.mean())\n        out[f\'ct_recon_{tag}_relative_density_error\'] = float(rec.mean() - volume.mean())\n    return out, recon_full, recon_sparse\n'
LIB_PATH.write_text(LIB_SOURCE, encoding="utf-8")
print(f"Saved {LIB_PATH.resolve()} ({LIB_PATH.stat().st_size/1024:.1f} kB)")
print("Main notebook:", MAIN_NOTEBOOK_NAME)
print("Paired test notebook:", TEST_NOTEBOOK_NAME)

## Cell 02 — User configuration
Edit only this cell for a new dataset. It writes `config.json`; all subsequent cells reload that file independently.

- `INPUT_PATH`: one STL/OBJ/3MF/etc. **or** a folder containing ordered 2D slice images.
- `MODE`: `core`, `extended`, or `exhaustive`. This notebook is designed for `exhaustive`.
- `SAVE_TRIPLET_IMAGES`: saves ABC and full 8-state grayscale images.
- `OPTIONAL_PERSISTENT_HOMOLOGY`: requires `gudhi`; if absent the PH cell writes a skip flag rather than failing.

In [ ]:
from pathlib import Path
import json

CONFIG = {
    "INPUT_PATH": r"CHANGE_ME/sample.stl",
    "WORKDIR": r"descriptor_run",
    "INPUT_KIND": "auto",              # auto | mesh | image_stack
    "VOXEL_SIZE_MM": 0.20,              # mesh mode
    "DX_MM": 0.20, "DY_MM": 0.20, "DZ_MM": 0.20,  # image-stack mode
    "THRESHOLD": 127, "INVERT": False,
    "Z_UPSAMPLE": 1,
    "MIN_COMPONENT_VOXELS": 1,
    "FILL_HOLES": False,
    "SAVE_TRIPLET_IMAGES": True,
    "TRIPLET_IMAGE_STRIDE": 1,
    "MAX_LAYER_LAG": 12,
    "MAX_PERSISTENCE_K": 7,
    "GRANULOMETRY_MAX_RADIUS": 8,
    "EULER_FILTRATION_MAX_RADIUS": 6,
    "SPECTRAL_MAX_DIM": 128,
    "OPTIONAL_PERSISTENT_HOMOLOGY": True,
    "PH_MAX_DIM_VOX": 64,
    "MODE": "exhaustive",
    # --- v4 additions ---
    "CURVATURE_SMOOTH_ITERATIONS": 12,
    "TORTUOSITY_MAX_SIDE": 48,
    "STRUT_GRAPH_MAX_SKELETON_VOXELS": 250000,
    "SOFT_TSPE_FALLBACK_BLUR_SIGMA": 0.8,
    "COMPUTE_CT_RECONSTRUCTION": True,
    "CT_RECON_N_ANGLES_FULL": 180,
    "CT_RECON_N_ANGLES_SPARSE": 60,
    "CT_RECON_ADD_POISSON_NOISE": False,
    # --- mesh-slicing robustness (STL/STEP 'jumping slice' fix) ---
    "MESH_REPAIR": True,                       # weld near-duplicate vertices + fix normals/holes before slicing
    "MESH_VOXELIZATION_BACKEND": "auto"        # auto | trimesh_fill | legacy_scanline
}

work = Path(CONFIG["WORKDIR"])
for d in [work, work/"checkpoints", work/"features", work/"images", work/"logs"]:
    d.mkdir(parents=True, exist_ok=True)
with open(work/"config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)
Path('.architected_descriptor_active_workdir.txt').write_text(str(work), encoding='utf-8')
print("Saved:", work/"config.json")
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))

## Cell 03 — Environment and dependency audit
Saves software versions to `checkpoints/00_environment.json`.

In [ ]:
from pathlib import Path
import json, sys, importlib
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run')
# If WORKDIR was changed, discover it from the local config file path manually here once.
config_path=work/"config.json"
if not config_path.exists():
    candidates=list(Path('.').glob('*/config.json'))
    if len(candidates)==1: config_path=candidates[0]; work=config_path.parent
cfg=json.loads(config_path.read_text(encoding='utf-8'))
mods=['numpy','pandas','scipy','skimage','trimesh','networkx','cv2','sklearn','gudhi']
env={'python':sys.version,'mode':cfg['MODE']}
for m in mods:
    try:
        mod=importlib.import_module(m); env[m]=getattr(mod,'__version__','installed')
    except Exception as e: env[m]=f'NOT_INSTALLED: {e}'
(work/'checkpoints').mkdir(parents=True,exist_ok=True)
(work/'checkpoints'/'00_environment.json').write_text(json.dumps(env,indent=2),encoding='utf-8')
print(json.dumps(env,indent=2))

## Cell 04 — Import geometry / image stack → raw voxel checkpoint
Output: `checkpoints/01_volume_raw.npz`, `checkpoints/01_input_meta.json`.

In [ ]:
from pathlib import Path
import json, sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); cfg=dl.load_json(work/'config.json'); inp=Path(cfg['INPUT_PATH'])
kind=cfg['INPUT_KIND']
if kind=='auto': kind='image_stack' if inp.is_dir() else 'mesh'
if kind=='image_stack':
    vol,spacing,meta=dl.load_image_stack(inp,cfg['THRESHOLD'],cfg['INVERT'],(cfg['DZ_MM'],cfg['DY_MM'],cfg['DX_MM']))
elif kind=='mesh':
    vol,spacing,meta=dl.load_mesh_as_voxels(inp,cfg['VOXEL_SIZE_MM'],
                                              backend=cfg.get('MESH_VOXELIZATION_BACKEND','auto'),
                                              repair=cfg.get('MESH_REPAIR',True),
                                              verbose=True)
else: raise ValueError(kind)
dl.save_volume(work/'checkpoints'/'01_volume_raw.npz',vol,spacing,meta)
dl.save_json(meta,work/'checkpoints'/'01_input_meta.json')
print('raw volume:',vol.shape,'spacing z,y,x [mm]:',spacing,'density:',vol.mean())

## Cell 05 — Preprocess / optional shape-based Z interpolation
Output: `checkpoints/02_volume_processed.npz`.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); cfg=dl.load_json(work/'config.json'); vol,spacing,meta=dl.load_volume(work/'checkpoints'/'01_volume_raw.npz')
vol=dl.preprocess_volume(vol,cfg['MIN_COMPONENT_VOXELS'],cfg['FILL_HOLES'])
vol,spacing=dl.shape_based_z_interpolation(vol,spacing,cfg['Z_UPSAMPLE'])
meta.update({'preprocessed':True,'z_upsample':cfg['Z_UPSAMPLE']})
dl.save_volume(work/'checkpoints'/'02_volume_processed.npz',vol,spacing,meta)
print('processed:',vol.shape,'spacing:',spacing,'density:',vol.mean())

## Cell 06 — Conventional 2D slice morphology
Per-slice: area fraction, perimeter, specific perimeter, component count, Euler/hole proxy, largest-component solidity/extent/eccentricity/orientation, equivalent diameter, axes, centroid, circularity, plus extensive z-profile statistics.

Outputs: `features/03_slice_table.csv`, `features/03_slice_summary.json`.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
df,s=dl.slice_descriptor_table(vol,spacing); df.to_csv(work/'features'/'03_slice_table.csv',index=False); dl.save_json(s,work/'features'/'03_slice_summary.json')
print('slice rows:',len(df),'summary descriptors:',len(s)); display(df.head())

## Cell 07 — Orthogonal 2D projection texture and moments
Adds GLCM texture statistics, Hu moments, entropy, and gradient statistics from XY/XZ/YZ occupancy and maximum projections.

Output: `features/04_projection_texture.json`.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
s=dl.projection_texture_descriptors(vol); dl.save_json(s,work/'features'/'04_projection_texture.json'); print('projection descriptors:',len(s))

## Cell 08 — Legacy / complementary 2-layer transition descriptors
Adjacent slices `(N,N+1)`: Jaccard, Dice, overlap coefficient, directional containment, symmetric difference, signed/absolute area change, centroid shift, and symmetric boundary displacement.

Outputs: `features/05_pair_table.csv`, `features/05_pair_summary.json`.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
df,s=dl.pair_descriptor_table(vol,spacing); df.to_csv(work/'features'/'05_pair_table.csv',index=False); dl.save_json(s,work/'features'/'05_pair_summary.json')
print('pair rows:',len(df),'summary descriptors:',len(s)); display(df.head())

## Cell 09 — Three-layer TSPE: A/B/C + complete 8-state encoding
For each `(N,N+1,N+2)` window:
- `A=111`, `B=110`, `C=011`
- all seven occupied 3-bit states retained
- state fractions and entropy
- A/B/C component morphology
- A↔B, A↔C, B↔C interface counts
- growth/decay, gap/re-entry, two-sided persistence

Outputs: `features/06_triplet_table.csv`, `features/06_triplet_summary.json`, optional grayscale images.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); cfg=dl.load_json(work/'config.json'); vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
outdir=work/'images'/'triplets' if cfg['SAVE_TRIPLET_IMAGES'] else None
df,s=dl.triplet_descriptor_table(vol,spacing,outdir,cfg['TRIPLET_IMAGE_STRIDE']); df.to_csv(work/'features'/'06_triplet_table.csv',index=False); dl.save_json(s,work/'features'/'06_triplet_summary.json')
print('triplet rows:',len(df),'summary descriptors:',len(s)); display(df.head())

## Cell 10 — Multi-layer persistence + multi-lag overlap
Generalizes the 3-layer concept to `k=2…K` consecutive layers and to lagged layer comparisons. Adds lagged Jaccard/Dice/symmetric-difference/mutual-information and overlap-decay descriptors.

Output: `features/07_multilayer_lag.json`.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); cfg=dl.load_json(work/'config.json'); vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
s={}; s.update(dl.multilevel_persistence_descriptors(vol,cfg['MAX_PERSISTENCE_K'])); s.update(dl.lag_overlap_descriptors(vol,cfg['MAX_LAYER_LAG']))
dl.save_json(s,work/'features'/'07_multilayer_lag.json'); print('multi-layer/lag descriptors:',len(s))

## Cell 11 — TSPE state-dynamics descriptors
Builds the complete `8 × 8 = 64` transition matrix between consecutive triplet-state images and adds state-specific self-persistence, transition entropy, A/B/C evolution, and all 64 transition probabilities.

Output: `features/08_tspe_transition.json`.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
s=dl.triplet_transition_descriptors(vol); dl.save_json(s,work/'features'/'08_tspe_transition.json'); print('TSPE dynamic descriptors:',len(s))

## Cell 12 — 3D global morphology / surface / shape moments
Relative density, volume, surface area, specific surface, compactness, sphericity, convex-hull ratios, centroid, radius of gyration, covariance/inertia eigenvalues, coordinate moments, surface-normal fabric, and dihedral-angle distribution.

Output: `features/09_morphology3d.json`.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
s=dl.morphology3d_descriptors(vol,spacing); dl.save_json(s,work/'features'/'09_morphology3d.json'); print('3D morphology descriptors:',len(s))

## Cell 13 — 3D topology, connectivity, Betti proxies, Euler filtration
Computes solid/void components at 6/18/26 connectivity, x/y/z spanning/percolation, Euler characteristics, enclosed cavities, Betti-number proxies, and a multiscale erosion/dilation Euler curve.

Output: `features/10_topology.json`.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); cfg=dl.load_json(work/'config.json'); vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
s=dl.topology_descriptors(vol); s.update(dl.euler_filtration_descriptors(vol,cfg['EULER_FILTRATION_MAX_RADIUS'])); dl.save_json(s,work/'features'/'10_topology.json'); print('topology descriptors:',len(s))

## Cell 14 — Local thickness, pore size, skeleton-sampled thickness, granulometry
Uses Euclidean distance transforms and morphological openings/closings over multiple scales.

Output: `features/11_size_granulometry.json`.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); cfg=dl.load_json(work/'config.json'); vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
s=dl.distance_thickness_descriptors(vol,spacing); s.update(dl.granulometry_descriptors(vol,cfg['GRANULOMETRY_MAX_RADIUS'])); dl.save_json(s,work/'features'/'11_size_granulometry.json'); print('size/granulometry descriptors:',len(s))

## Cell 15 — Classical spatial statistics
Includes solid/void chord-length distributions, lineal-path survival, directional two-point correlation, correlation lengths, and selected L-shaped 3-point correlations.

Output: `features/12_spatial_statistics.json`.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
s={}; s.update(dl.chord_descriptors(vol,spacing)); s.update(dl.lineal_path_descriptors(vol)); s.update(dl.two_point_axis_descriptors(vol)); s.update(dl.three_point_descriptors(vol)); dl.save_json(s,work/'features'/'12_spatial_statistics.json'); print('spatial-stat descriptors:',len(s))

## Cell 16 — Multiscale box counting, fractal dimension, lacunarity
Calculated independently for solid, void, and interface/boundary phases over multiple box sizes.

Output: `features/13_multiscale_fractal.json`.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
s=dl.multiscale_descriptors(vol); dl.save_json(s,work/'features'/'13_multiscale_fractal.json'); print('multiscale/fractal descriptors:',len(s))

## Cell 17 — 3D Fourier/spectral + directional/MIL descriptors
3D FFT power entropy, dominant wavelength, radial spectrum bins, wave-vector anisotropy, directional occupancy, mean intercept length (MIL), and directional phase-transition distributions.

Output: `features/14_spectral_directional.json`.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); cfg=dl.load_json(work/'config.json'); vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
s=dl.spectral_descriptors(vol,spacing,cfg['SPECTRAL_MAX_DIM']); s.update(dl.directional_descriptors(vol,spacing)); dl.save_json(s,work/'features'/'14_spectral_directional.json'); print('spectral/directional descriptors:',len(s))

## Cell 18 — Skeleton / network / graph-spectral descriptors
Extracts a 3D skeleton and computes length proxy, voxel-degree distribution, endpoints, branch points, graph cycle-rank proxy, degree entropy, and—when graph size permits—Laplacian algebraic connectivity and adjacency spectral radius.

Output: `features/15_skeleton_network.json`.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
s=dl.skeleton_network_descriptors(vol,spacing); dl.save_json(s,work/'features'/'15_skeleton_network.json'); print('skeleton/network descriptors:',len(s))

## Cell 19 — Optional persistent homology
If `gudhi` is installed, computes cubical-complex persistence summaries for dimensions 0, 1, and 2. If not installed, the checkpoint records `persistent_homology_available = 0` and the rest of the notebook continues normally.

Output: `features/16_persistent_homology.json`.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); cfg=dl.load_json(work/'config.json'); vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
if cfg['OPTIONAL_PERSISTENT_HOMOLOGY']: s=dl.persistent_homology_descriptors(vol,cfg['PH_MAX_DIM_VOX'])
else: s={'persistent_homology_available':0,'persistent_homology_disabled_by_config':1}
dl.save_json(s,work/'features'/'16_persistent_homology.json'); print(s)

## Cell 19b — [v4 NEW] Grayscale-preserving slice stack (soft-TSPE input)

The v3 pipeline binarizes every slice on load (`load_image_stack`: `im>threshold`) before any A/B/C overlap logic ever runs, so the "grayscale-based overlap" idea in the original concept never actually reached the descriptor stage. This cell re-reads the same slice folder WITHOUT binarizing, keeping normalized [0,1] intensity per voxel, so Cell 19c below can compute the real fuzzy/soft version of the A/B/C triplet encoding. Skipped gracefully for mesh input (no native grayscale slices exist); Cell 19c then falls back to a lightly blurred copy of the binary volume.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run')
cfg=dl.load_json(work/'config.json')
gpath=work/'checkpoints'/'01b_volume_grayscale.npz'
inp=Path(cfg['INPUT_PATH']); kind=cfg['INPUT_KIND']
if kind=='auto': kind='image_stack' if inp.is_dir() else 'mesh'
if kind=='image_stack':
    gvol,gspacing,gmeta=dl.load_image_stack_grayscale(inp,(cfg['DZ_MM'],cfg['DY_MM'],cfg['DX_MM']),cfg['INVERT'])
    dl.save_grayscale_stack(gpath,gvol,gspacing,gmeta)
    print('grayscale stack saved:',gvol.shape,'dtype',gvol.dtype,'range',(float(gvol.min()),float(gvol.max())))
else:
    print('INPUT_KIND is mesh -> no native grayscale slices; Cell 19c will use a blurred-binary surrogate instead.')


## Cell 19c — [v4 NEW] Grayscale soft-TSPE (fuzzy A/B/C overlap, partial-volume diagnostics)

Continuous-valued counterpart of Cell 09's hard TSPE. For consecutive grayscale slices I_N, I_{N+1}, I_{N+2} in [0,1]: `A_soft=min(I_N,I_{N+1},I_{N+2})`, `B_soft=min(I_N,I_{N+1},1-I_{N+2})`, `C_soft=min(I_{N+1},I_{N+2},1-I_N)` (Zadeh fuzzy-AND). Also reports how much information hard thresholding would have discarded (`soft_A_hard_agreement_dice`, `soft_*_partial_volume_fraction`). Uses the native-resolution grayscale checkpoint from Cell 19b when available (i.e. real acquired slices, not z-interpolated ones -- fabricating soft/grayscale values for an interpolated slice would not be physically meaningful), otherwise a blurred copy of the processed binary volume.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run')
cfg=dl.load_json(work/'config.json')
gpath=work/'checkpoints'/'01b_volume_grayscale.npz'
if gpath.exists():
    gvol,gspacing,_=dl.load_grayscale_stack(gpath)
else:
    vol,gspacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
    gvol=dl.grayscale_stack_from_binary(vol, blur_sigma=cfg.get('SOFT_TSPE_FALLBACK_BLUR_SIGMA',0.8))
outdir=work/'images'/'soft_triplets' if cfg['SAVE_TRIPLET_IMAGES'] else None
df,s=dl.soft_triplet_descriptor_table(gvol,gspacing,outdir,cfg['TRIPLET_IMAGE_STRIDE'])
df.to_csv(work/'features'/'v4_01_soft_triplet_table.csv',index=False)
dl.save_json(s,work/'features'/'v4_01_soft_triplet_summary.json')
print('soft-triplet rows:',len(df),'summary descriptors:',len(s)); display(df.head())


## Cell 19d — [v4 NEW] Discrete surface curvature (mean H, Gaussian K, shape index, curvedness)

Was completely absent from v3 (only edge-level dihedral-angle statistics existed, not true per-vertex curvature). Cotangent-Laplacian mean curvature + angle-defect Gaussian curvature on a Taubin-smoothed marching-cubes surface (Meyer et al. 2003), reduced to principal curvatures k1/k2, Koenderink shape index/curvedness, and -- most useful for telling architected-material topologies apart -- the area-weighted elliptic (dome/node) vs. hyperbolic (saddle/TPMS-sheet) vs. parabolic (cylindrical/strut) surface-fraction split. Pure numpy/scipy/scikit-image; does not require trimesh.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run')
cfg=dl.load_json(work/'config.json')
vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
s=dl.curvature_descriptors(vol,spacing,smooth_iterations=cfg.get('CURVATURE_SMOOTH_ITERATIONS',12))
dl.save_json(s,work/'features'/'v4_02_curvature.json')
print('curvature descriptors:',len(s))


## Cell 19e — [v4 NEW] Transport tortuosity (solid & void, geodesic vs. straight-line)

Also absent from v3. tau = <geodesic path length> / <straight-line length> from the low-coordinate face to the high-coordinate face along each axis, computed with multi-source Dijkstra on the 6-connected voxel graph of the largest connected component of each phase (downsampled to `TORTUOSITY_MAX_SIDE` per dimension for tractability). NaN when a phase does not span/percolate that axis. Void tortuosity is the standard permeability/diffusivity descriptor; solid tortuosity is the analogous descriptor for conduction path length through the strut network.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run')
cfg=dl.load_json(work/'config.json')
vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
s=dl.tortuosity_descriptors(vol,spacing,max_side=cfg.get('TORTUOSITY_MAX_SIDE',48))
dl.save_json(s,work/'features'/'v4_03_tortuosity.json')
print('tortuosity descriptors:',len(s)); print(s)


## Cell 19f — [v4 NEW] Strut/node lattice graph (coordination number, strut length/tortuosity/thickness)

Cell 18's skeleton family only reports coarse, per-voxel degree statistics. This cell reduces the 26-connected skeleton voxel graph to the actual node/strut graph used in architected-material literature: junction nodes (skeleton voxels of degree != 2) connected by struts (chains of degree-2 voxels). Reports nodal coordination number Z (Maxwell isostaticity reference: Z=6 for a 3D central-force frame), per-strut geodesic/straight length and tortuosity, and per-strut thickness (sampled from the solid distance-transform along the path) including a thickness-uniformity CV.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run')
cfg=dl.load_json(work/'config.json')
vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
s=dl.strut_graph_descriptors(vol,spacing,max_skeleton_voxels=cfg.get('STRUT_GRAPH_MAX_SKELETON_VOXELS',250000))
dl.save_json(s,work/'features'/'v4_04_strut_graph.json')
print('strut/node graph descriptors:',len(s))


## Cell 19g — [v4 NEW] X-ray CT projection simulation + filtered back-projection reconstruction

Implements the "use real X-ray/CT reconstruction techniques as an alternative extraction pathway" request literally: forward-projects every axial slice with the Radon transform at `CT_RECON_N_ANGLES_FULL` angles (optionally injecting Poisson photon-counting noise, Beer-Lambert attenuation model), reconstructs with filtered back-projection (ramp filter), and repeats at `CT_RECON_N_ANGLES_SPARSE` angles to show the effect of a lower-dose acquisition. Reports reconstruction fidelity (Dice/IoU/relative-density error) against this notebook's own direct slice-stack volume, and re-runs the thickness descriptors on the full-angle CT-reconstructed volume (prefixed `ct_recon_`) so the two independent 3D-reconstruction pipelines are directly comparable on the same structural descriptor. A user who only has raw radiographic projections (rather than already-registered slice images) can call `dl.xray_ct_projection_reconstruction` directly and skip the slice-stack step (Cells 04-05) entirely.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run')
cfg=dl.load_json(work/'config.json')
vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
if cfg.get('COMPUTE_CT_RECONSTRUCTION', True):
    s,rec_full,rec_sparse=dl.xray_ct_projection_reconstruction(
        vol, spacing,
        n_angles=cfg.get('CT_RECON_N_ANGLES_FULL',180),
        sparse_n_angles=cfg.get('CT_RECON_N_ANGLES_SPARSE',60),
        add_poisson_noise=cfg.get('CT_RECON_ADD_POISSON_NOISE',False))
    thick_ct=dl.distance_thickness_descriptors(rec_full,spacing)
    s.update({f'ct_recon_{k}':v for k,v in thick_ct.items()})
    dl.save_volume(work/'checkpoints'/'v4_ct_recon_full_angle.npz', rec_full, spacing, {'source_type':'ct_recon_full_angle'})
    dl.save_json(s,work/'features'/'v4_05_ct_recon_fidelity.json')
    print('CT reconstruction descriptors:',len(s))
else:
    print('CT reconstruction skipped by config (COMPUTE_CT_RECONSTRUCTION=False)')


## Cell 20 — Aggregate the complete descriptor pool
Reads every JSON feature checkpoint from disk and merges them into one ML-ready row. No feature selection is performed here.

Outputs:
- `features/descriptors_ALL.csv` — one row per structure
- `features/descriptor_catalog.csv` — family mapping
- `features/descriptor_QA.csv` — finite/NaN/Inf audit

In [ ]:
from pathlib import Path
import json, sys, numpy as np, pandas as pd
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); feat=work/'features'; merged={}
for p in sorted(feat.glob('*.json')):
    d=dl.load_json(p)
    for k,v in d.items():
        if isinstance(v,(int,float,bool)) or v is None: merged[k]=v
meta=dl.load_json(work/'checkpoints'/'01_input_meta.json')
merged={'sample_id':Path(meta.get('source','sample')).stem,**merged}
pd.DataFrame([merged]).to_csv(feat/'descriptors_ALL.csv',index=False)
catalog=dl.descriptor_catalog_from_names([k for k in merged if k!='sample_id']); catalog.to_csv(feat/'descriptor_catalog.csv',index=False)
qa=[]
for k,v in merged.items():
    if k=='sample_id': continue
    try: x=float(v); state='finite' if np.isfinite(x) else ('nan' if np.isnan(x) else 'inf')
    except Exception: state='non_numeric'
    qa.append({'descriptor':k,'state':state,'value':v})
pd.DataFrame(qa).to_csv(feat/'descriptor_QA.csv',index=False)
print('TOTAL DESCRIPTORS:',len(merged)-1)
print(pd.Series([r['state'] for r in qa]).value_counts())
display(pd.DataFrame([merged]).iloc[:,:20])

## Cell 21 — Save a compact run manifest
This records which checkpoints exist and their descriptor counts, making runs auditable and restartable.

In [ ]:
from pathlib import Path
import json, pandas as pd
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); records=[]
for p in sorted((work/'features').glob('*')):
    rec={'file':p.name,'bytes':p.stat().st_size}
    if p.suffix=='.json':
        try: rec['feature_count']=len(json.loads(p.read_text(encoding='utf-8')))
        except: pass
    elif p.suffix=='.csv':
        try: rec['rows'],rec['columns']=pd.read_csv(p).shape
        except: pass
    records.append(rec)
pd.DataFrame(records).to_csv(work/'run_manifest.csv',index=False)
display(pd.DataFrame(records))

## Cell 22 — Optional reconstructed STL export
Exports the processed voxel field via marching cubes. This is a CT-like virtual reconstruction derived from the same slice stack used for TSPE.

In [ ]:
from pathlib import Path
import sys
_cwd=Path.cwd().resolve(); _cands=[_cwd,_cwd/'Code',_cwd.parent,_cwd.parent/'Code']; _code_dir=next((d.resolve() for d in _cands if (d/'descriptor_library.py').is_file()),_cwd); sys.path.insert(0,str(_code_dir)) if str(_code_dir) not in sys.path else None
import descriptor_library as dl
ptr=Path('.architected_descriptor_active_workdir.txt'); work=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run'); vol,spacing,_=dl.load_volume(work/'checkpoints'/'02_volume_processed.npz')
trimesh_missing=False
try:
    mesh=dl.marching_mesh(vol,spacing)
except RuntimeError as e:
    mesh=None; trimesh_missing=True; print('STL export skipped:',e)
if mesh is not None:
    path=work/'reconstructed_from_slices.stl'; mesh.export(path); print('saved',path,'faces=',len(mesh.faces))
elif not trimesh_missing:
    print('Reconstruction skipped: empty/full volume')

## Cell 23 — Batch aggregation across many completed runs
If you process multiple structures into separate run folders, list those folders below. This cell combines their `descriptors_ALL.csv` files into a single training matrix **without selecting features**.

In [ ]:
from pathlib import Path
import pandas as pd
ptr=Path('.architected_descriptor_active_workdir.txt')
active=Path(ptr.read_text(encoding='utf-8').strip()) if ptr.exists() else Path('descriptor_run')
RUN_FOLDERS = [active]  # add Path('run_structure_02'), ...
frames=[]
for r in RUN_FOLDERS:
    p=r/'features'/'descriptors_ALL.csv'
    if p.exists(): frames.append(pd.read_csv(p))
if not frames: raise FileNotFoundError('No descriptors_ALL.csv found in RUN_FOLDERS')
master=pd.concat(frames,ignore_index=True,sort=False)
master.to_csv('MASTER_descriptors_ALL.csv',index=False)
print('master shape:',master.shape); display(master.head())

## Descriptor families intentionally retained before feature selection
This notebook deliberately keeps correlated and partially redundant descriptors. Examples include Jaccard/Dice/overlap, density/surface/compactness, multiple percentile statistics, several connectivity definitions, TSPE state fractions, multi-lag correlations, chord/lineal-path/two-point statistics, and both local-thickness and granulometric scale descriptors.

Recommended later feature-selection sequence (in a separate notebook/code, as requested):
1. remove invalid/constant descriptors,
2. quantify pairwise correlation and VIF,
3. stability selection / mutual information / mRMR,
4. tree/boosting permutation importance + SHAP,
5. recursive feature elimination under grouped cross-validation,
6. retain a physically interpretable Pareto set rather than only the numerically smallest subset.